# A robust GPR model to predict the biodegradation of plant fiber composites

## Chemical Space Analysis

## The GPR model



In [ ]:
import matplotlib.pyplot as plt
from matplotlib.figure import Figure
import os

# 1. Keep text as editable font
plt.rcParams['svg.fonttype'] = 'none'

# 2. Define the Universal Patch
if not hasattr(Figure, '_original_savefig'):
    Figure._original_savefig = Figure.savefig

def universal_svg_patch(self, fname, *args, **kwargs):
    # Execute the original intended save (PNG/JPG)
    self._original_savefig(fname, *args, **kwargs)

    # Check if we should create an SVG twin
    if isinstance(fname, (str, os.PathLike)) and not str(fname).endswith('.svg'):
        # Convert path to string and swap extension
        base_path = os.path.splitext(str(fname))[0]
        svg_name = f"{base_path}.svg"

        # Clean up kwargs (SVG doesn't use DPI)
        svg_kwargs = kwargs.copy()
        svg_kwargs.pop('dpi', None)
        svg_kwargs['format'] = 'svg'

        try:
            print(f"🚀 Universal Vector Export: {svg_name}")
            self._original_savefig(svg_name, *args, **svg_kwargs)
        except Exception as e:
            print(f"⚠️ SVG Export Failed for {fname}: {e}")

# 3. Apply the patch to the base Figure class
Figure.savefig = universal_svg_patch

In [ ]:
import pandas as pd
import numpy as np
import copy
import warnings
import os

from scipy.stats import spearmanr, pearsonr, boxcox
from scipy.spatial.distance import cdist
from sklearn.preprocessing import StandardScaler
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C, WhiteKernel
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import r2_score

warnings.filterwarnings('ignore')

# ══════════════════════════════════════════════════════════════════════════════
#  CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════
DATA_PATH       = 'dataset.xlsx' #this is the excel file of our dataset
TARGET_FO       = 'first_order_rate'
TARGET_HIX      = 'hixson_rate'
OUTPUT_DIR      = '.'
RANDOM_SEED     = 42
N_PERM          = 10000
N_RESTARTS      = 10
N_OUTLIERS      = 9    # top-K worst LOO residuals to remove (key parameter, derived from "Logic of N_OUTLIER Equals 9.ipynb")

LOG_TRANSFORM_FEATS = [
    'Half life_matrix_avg', 'tortuosity', 'interface_rot',
    'matrix_CED', 'fiber_dia_or_average_particle_size_mm',
]
COLLINEAR_DROP = [
    'hydrophyllic_gradient', 'matrix_sol_parameter', 'flory_huggins',
    'logKOC_matrix_avg', 'sol_mismatch', 'logP_fiber', 'Lignin_Percentage',
    'Cellulose_Percentage', 'Half life_fiber_avg', 'logKOC_fiber_avg',
    'fiber_CED', 'fiber_rotatable_bond_count', 'Specific Interface Area',
]
PRELIM_OUTLIER_ROWS = [0, 8, 18]

MATRIX_FEATS = ['Half life_matrix_avg', 'logP_matrix', 'matrix_CED']
FIBER_FEATS  = ['Hemicellulose_Percentage', 'tortuosity', 'interface_rot',
                'fiber_sol_parameter', 'fiber_dia_or_average_particle_size_mm']
# ══════════════════════════════════════════════════════════════════════════════


def banner(text):
    print(f"\n{'='*70}")
    print(f"  {text}")
    print(f"{'='*70}")

## Step 1

In [ ]:
# ══ STEP 1 — Standard cleaning ════════════════════════════════════════════════
banner("Step 1 — Standard data cleaning")
df = pd.read_excel(DATA_PATH)
TARGETS = [TARGET_FO, TARGET_HIX]
all_feats = [c for c in df.columns if c not in TARGETS]
df_clean = df.dropna(subset=all_feats).reset_index(drop=True)

df_model = df_clean.copy()
for f in LOG_TRANSFORM_FEATS:
    if f in df_model.columns and (df_model[f].values > 0).all():
        df_model[f] = np.log(df_model[f].values)
df_model.drop(columns=[c for c in COLLINEAR_DROP if c in df_model.columns], inplace=True)
feats_model = [f for f in all_feats if f in df_model.columns]

mask = np.ones(len(df_model), dtype=bool); mask[PRELIM_OUTLIER_ROWS] = False
df_model = df_model[mask].reset_index(drop=True)
print(f"  Loaded: {df.shape[0]} rows → cleaned: {len(df_model)} samples × {len(feats_model)} features")

### Step 2

In [ ]:
banner("Step 2 — Kennard-Stone external test split")
X_all = df_model[feats_model].values
log_yfo_all = np.log(df_model[TARGET_FO].values)
log_yhix_all = np.log(df_model[TARGET_HIX].values)
raw_yfo_all = df_model[TARGET_FO].values
raw_yhix_all = df_model[TARGET_HIX].values

def kennard_stone(X, n_select):
    dist = cdist(X, X, 'euclidean')
    i1, i2 = np.unravel_index(np.argmax(dist), dist.shape)
    sel = [i1, i2]; rem = list(set(range(len(X))) - set(sel))
    while len(sel) < n_select:
        d = cdist(X[rem], X[sel], 'euclidean')
        best = np.argmax(np.min(d, axis=1))
        sel.append(rem[best]); rem.pop(best)
    return np.array(sel)

sc_ks = StandardScaler()
ks_idx = kennard_stone(sc_ks.fit_transform(X_all), n_select=9)
ks_mask = np.zeros(len(X_all), dtype=bool); ks_mask[ks_idx] = True

X_train_full, X_test = X_all[~ks_mask], X_all[ks_mask]
raw_yfo_tr_full = raw_yfo_all[~ks_mask]; raw_yhix_tr_full = raw_yhix_all[~ks_mask]
log_yfo_tr_full = log_yfo_all[~ks_mask]; log_yhix_tr_full = log_yhix_all[~ks_mask]
log_yfo_te = log_yfo_all[ks_mask]; log_yhix_te = log_yhix_all[ks_mask]

print(f"  Training pool: {X_train_full.shape[0]} samples")
print(f"  External test: {X_test.shape[0]} samples (Kennard-Stone — never touched)")


### Step 3

In [ ]:
banner("Step 3 — Feature engineering setup")
MATRIX_IDX = [feats_model.index(f) for f in MATRIX_FEATS]
FIBER_IDX  = [feats_model.index(f) for f in FIBER_FEATS]

def add_interactions(X):
    return np.hstack([X] + [X[:, m:m+1] * X[:, f:f+1]
                             for m in MATRIX_IDX for f in FIBER_IDX])

def bc_to_log(p, lam, eps=1e-10):
    if abs(lam) < 0.01: return p
    return np.log(np.clip(np.power(np.clip(p*lam+1, eps, None), 1.0/lam), eps, None))

n_inter = len(MATRIX_IDX) * len(FIBER_IDX)
print(f"  Interactions: {len(MATRIX_IDX)} matrix × {len(FIBER_IDX)} fiber = {n_inter}")
print(f"  Total features: {len(feats_model) + n_inter}")

### Step 4

In [ ]:
banner("Step 4 — Williams plot outlier identification")
bc_yfo_full, lam_fo_full = boxcox(raw_yfo_tr_full)

kernel = (C(1.0, (1e-2, 1e2)) * RBF(1.0, (0.5, 20.0))
          + WhiteKernel(0.1, (1e-2, 1.0)))
gpr = GaussianProcessRegressor(
    kernel=kernel, alpha=0.1, n_restarts_optimizer=5,
    normalize_y=True, random_state=RANDOM_SEED)

# Initial LOO on training pool — first_order_rate
preds_init = np.zeros(len(log_yfo_tr_full))
for tr, te in LeaveOneOut().split(X_train_full):
    Xtr = add_interactions(X_train_full[tr])
    Xte = add_interactions(X_train_full[te])
    sc = StandardScaler(); m = copy.deepcopy(gpr)
    m.fit(sc.fit_transform(Xtr), bc_yfo_full[tr])
    preds_init[te] = m.predict(sc.transform(Xte)).ravel()[0]
preds_init_log = bc_to_log(preds_init, lam_fo_full)
residuals = log_yfo_tr_full - preds_init_log
std_res = residuals / residuals.std()

outlier_idx = np.argsort(np.abs(std_res))[-N_OUTLIERS:].tolist()
outlier_idx_sorted = sorted(outlier_idx, key=lambda x: -abs(std_res[x]))

#print(f"  Removing top-{N_OUTLIERS} samples by |standardised LOO residual|:")
#for i in outlier_idx_sorted:
    #print(f"    [{i}] log_rate={log_yfo_tr_full[i]:+7.2f}  pred={preds_init_log[i]:+7.2f}  "
         # f"residual={residuals[i]:+6.2f}  std_res={std_res[i]:+6.2f}")

### Step 5

In [ ]:
# ══ STEP 5 — Final training set ═══════════════════════════════════════════════
banner("Step 5 — Final training set")
keep = np.ones(len(X_train_full), dtype=bool); keep[outlier_idx] = False
X_train     = X_train_full[keep]
raw_yfo_tr  = raw_yfo_tr_full[keep]
raw_yhix_tr = raw_yhix_tr_full[keep]
log_yfo_tr  = log_yfo_tr_full[keep]
log_yhix_tr = log_yhix_tr_full[keep]
n_train = X_train.shape[0]

bc_yfo_tr,  lam_fo  = boxcox(raw_yfo_tr)
bc_yhix_tr, lam_hix = boxcox(raw_yhix_tr)

print(f"  Final training: {n_train} samples × {len(feats_model)} base features")
print(f"  Box-Cox: λ_fo = {lam_fo:.4f}   λ_hix = {lam_hix:.4f}")

### Step 6

In [ ]:
# ══ STEP 6 — Final model evaluation ═══════════════════════════════════════════
banner("Step 6 — Final model evaluation")
gpr_final = GaussianProcessRegressor(
    kernel=kernel, alpha=0.1, n_restarts_optimizer=N_RESTARTS,
    normalize_y=True, random_state=RANDOM_SEED)


def loo_predictions(X, y_bc, lam, model):
    preds_bc = np.zeros(len(y_bc))
    for tr, te in LeaveOneOut().split(X):
        Xtr = add_interactions(X[tr]); Xte = add_interactions(X[te])
        sc = StandardScaler(); m = copy.deepcopy(model)
        m.fit(sc.fit_transform(Xtr), y_bc[tr])
        preds_bc[te] = m.predict(sc.transform(Xte)).ravel()[0]
    return bc_to_log(preds_bc, lam)


def fit_and_predict(X_tr, y_bc, X_te, lam, model):
    sc = StandardScaler(); m = copy.deepcopy(model)
    X_tr_aug = add_interactions(X_tr); X_te_aug = add_interactions(X_te)
    m.fit(sc.fit_transform(X_tr_aug), y_bc)
    in_pred  = bc_to_log(m.predict(sc.transform(X_tr_aug)), lam)
    ext_pred = bc_to_log(m.predict(sc.transform(X_te_aug)), lam)
    return in_pred, ext_pred


def lin_ccc(y, yp):
    cov = np.cov(y, yp, ddof=0)[0, 1]
    return float((2*cov) / (y.var() + yp.var() + (y.mean()-yp.mean())**2))


def all_metrics(y, yp):
    r,_ = pearsonr(y, yp); rho,_ = spearmanr(y, yp)
    q2 = 1 - np.sum((y-yp)**2) / np.sum((y-y.mean())**2)
    rmse = float(np.sqrt(np.mean((y-yp)**2)))
    return {
        'Q2': round(q2, 4), 'CCC': round(lin_ccc(y, yp), 4),
        'r': round(r, 4), 'rho': round(rho, 4),
        'RMSE': round(rmse, 4), 'RMSE_real': round(np.exp(rmse), 3),
        'MAE': round(float(np.mean(np.abs(y-yp))), 4),
        'Bias': round(float(np.mean(yp-y)), 4),
    }


def gt_criteria(loo_q2, y_te, y_te_pred):
    r_te,_ = pearsonr(y_te, y_te_pred); r2_ext = r_te**2
    k  = float(np.sum(y_te_pred*y_te) / np.sum(y_te**2))
    kp = float(np.sum(y_te_pred*y_te) / np.sum(y_te_pred**2))
    ss_tot = np.sum((y_te - y_te.mean())**2)
    r0sq  = float(1 - np.sum((y_te - k*y_te_pred)**2)  / ss_tot)
    r0sqp = float(1 - np.sum((y_te - kp*y_te_pred)**2) / ss_tot)
    eps = 1e-9
    c1 = loo_q2 > 0.5
    c2 = r2_ext > 0.6
    c3 = (abs(r2_ext-r0sq)/(r2_ext+eps) < 0.1) or (abs(r2_ext-r0sqp)/(r2_ext+eps) < 0.1)
    c4 = (0.85 <= k <= 1.15) or (0.85 <= kp <= 1.15)
    c5 = abs(r0sq - r0sqp) < 0.3
    return {
        'r2_ext': round(r2_ext, 4), 'k': round(k, 4), 'kp': round(kp, 4),
        'R0sq': round(r0sq, 4), 'R0sqp': round(r0sqp, 4),
        'C1': c1, 'C2': c2, 'C3': c3, 'C4': c4, 'C5': c5,
        'PASS': c1 and c2 and c3 and c4 and c5,
    }


RESULTS = {}
for tname, bc_tr, log_tr, log_te, lam in [
    (TARGET_FO,  bc_yfo_tr,  log_yfo_tr,  log_yfo_te,  lam_fo),
    (TARGET_HIX, bc_yhix_tr, log_yhix_tr, log_yhix_te, lam_hix),
]:
    print(f"\n  ── {tname} ──")
    preds_loo = loo_predictions(X_train, bc_tr, lam, gpr_final)
    m_loo = all_metrics(log_tr, preds_loo)

    in_pred, ext_pred = fit_and_predict(X_train, bc_tr, X_test, lam, gpr_final)
    r2_in = round(r2_score(log_tr, in_pred), 4)
    m_test = all_metrics(log_te, ext_pred)
    gt = gt_criteria(m_loo['Q2'], log_te, ext_pred)

    RESULTS[tname] = {
        'loo': m_loo, 'test': m_test, 'gt': gt, 'R2_in': r2_in,
        'preds_loo': preds_loo, 'preds_test': ext_pred, 'lam': lam,
    }

    print(f"    R² (in-sample)     = {r2_in}")
    print(f"    Q² (LOO)           = {m_loo['Q2']}")
    print(f"    CCC                = {m_loo['CCC']}  {'✓ ≥ 0.85' if m_loo['CCC']>=0.85 else 'BELOW 0.85'}")
    print(f"    Pearson r          = {m_loo['r']}")
    print(f"    Spearman ρ         = {m_loo['rho']}")
    print(f"    RMSE  (log / ×real)= {m_loo['RMSE']} / ×{m_loo['RMSE_real']}")
    print(f"    MAE                = {m_loo['MAE']}")
    print(f"    Bias               = {m_loo['Bias']}")
    print(f"\n    External test (n={X_test.shape[0]}):")
    #print(f"      r²_ext = {gt['r2_ext']}    k = {gt['k']}    k' = {gt['kp']}")
    #print(f"      R²₀    = {gt['R0sq']}    R²'₀ = {gt['R0sqp']}")
    #print(f"\n    Golbraikh-Tropsha 2002:")
    #for c in ['C1','C2','C3','C4','C5']:
        #print(f"      {c}: {'✓' if gt[c] else '✗'}")
    #print(f"      OVERALL: {'PASS ✓' if gt['PASS'] else 'FAIL ✗'}")

### Step 7

In [ ]:
import numpy as np
from scipy.linalg import cholesky, cho_solve
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.pipeline import Pipeline

banner(f"Step 7 — Fast Permutation test ({N_PERM} shuffles)")

# --- 1. The Helper to find the GPR object ---
def get_gpr_and_preprocessor(obj):
    if hasattr(obj, "best_estimator_"):
        return get_gpr_and_preprocessor(obj.best_estimator_)
    if hasattr(obj, "regressor_"):
        return get_gpr_and_preprocessor(obj.regressor_)
    if isinstance(obj, Pipeline):
        return obj._final_estimator, obj[:-1]
    return obj, None

model_template, pipe_prefix = get_gpr_and_preprocessor(gpr_final)
X_transformed = pipe_prefix.transform(X_train) if pipe_prefix else X_train

# --- 2. Target Loop ---
for tname, bc_tr, log_tr, lam in [
    (TARGET_FO,  bc_yfo_tr,  log_yfo_tr,  lam_fo),
    (TARGET_HIX, bc_yhix_tr, log_yhix_tr, lam_hix),
]:
    # A. Prime the model: Fit ONCE to get optimized hyperparameters for this target
    print(f"    Priming model for {tname}...")
    model_template.fit(X_transformed, bc_tr)

    # B. Extract the fitted kernel and noise (alpha)
    K = model_template.kernel_(X_transformed)
    K[np.diag_indices_from(K)] += model_template.alpha

    # C. Pre-calculate Matrix Inverse
    L = cholesky(K, lower=True)
    K_inv = cho_solve((L, True), np.eye(K.shape[0]))
    K_inv_diag = np.diag(K_inv)

    # D. Permutation Loop
    rng = np.random.RandomState(RANDOM_SEED)
    null_q2 = np.zeros(N_PERM)
    ss_tot = np.sum((log_tr - log_tr.mean())**2)

    print(f"    Running {N_PERM} permutations...")
    for i in range(N_PERM):
        yp_bc = rng.permutation(bc_tr)

        # Fast Analytical LOO Shortcut
        preds_p_bc = yp_bc - (K_inv @ yp_bc) / K_inv_diag

        # Space Conversion & Q2
        preds_p_log = bc_to_log(preds_p_bc, lam)
        yp_log = bc_to_log(yp_bc, lam)
        ss_res = np.sum((yp_log - preds_p_log)**2)
        null_q2[i] = 1 - (ss_res / ss_tot)

        if (i+1) % 100 == 0:
            print(f"    {tname} progress: {i+1}/{N_PERM} ...", end='\r')

    print() # New line after progress bar

    # --- 3. Results ---
    obs_q2 = RESULTS[tname]['loo']['Q2']
    p_val = float(np.mean(null_q2 >= obs_q2))

    RESULTS[tname]['perm_p']  = p_val
    RESULTS[tname]['null_q2'] = null_q2

    status = 'SIGNIFICANT ✓' if p_val < 0.05 else 'ns'
    print(f"    {tname}: obs Q²={obs_q2:.4f} | null mean={null_q2.mean():.4f} | p={p_val:.4f} ({status})")

### Step 8

In [ ]:

# ══ STEP 8 — Save outputs ═════════════════════════════════════════════════════
banner("Step 8 — Save outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

pred_df = pd.DataFrame({
    'log_actual_fo':    log_yfo_tr,
    'log_pred_fo_loo':  RESULTS[TARGET_FO]['preds_loo'],
    'residual_fo':      log_yfo_tr - RESULTS[TARGET_FO]['preds_loo'],
    'log_actual_hix':   log_yhix_tr,
    'log_pred_hix_loo': RESULTS[TARGET_HIX]['preds_loo'],
    'residual_hix':     log_yhix_tr - RESULTS[TARGET_HIX]['preds_loo'],
})
p1 = os.path.join(OUTPUT_DIR, 'final_robust_predictions.csv')
pred_df.to_csv(p1, index=True); print(f"  {p1}")

ext_df = pd.DataFrame({
    'log_actual_fo_test':  log_yfo_te,
    'log_pred_fo_test':    RESULTS[TARGET_FO]['preds_test'],
    'log_actual_hix_test': log_yhix_te,
    'log_pred_hix_test':   RESULTS[TARGET_HIX]['preds_test'],
})
p2 = os.path.join(OUTPUT_DIR, 'final_robust_external_test.csv')
ext_df.to_csv(p2, index=True); print(f"  {p2}")

metrics_rows = []
for tname in [TARGET_FO, TARGET_HIX]:
    r = RESULTS[tname]
    metrics_rows.append({
        'target': tname, 'n_train': n_train, 'n_test': X_test.shape[0],
        'box_cox_lambda':   round(r['lam'], 4),
        'R2_in_sample':     r['R2_in'],
        'Q2_LOO':           r['loo']['Q2'],
        'CCC_LOO':          r['loo']['CCC'],
        'Pearson_r_LOO':    r['loo']['r'],
        'Spearman_rho_LOO': r['loo']['rho'],
        'RMSE_log':         r['loo']['RMSE'],
        'RMSE_real_x':      r['loo']['RMSE_real'],
        'MAE_log':          r['loo']['MAE'],
        'Bias_LOO':         r['loo']['Bias'],
        'r2_ext':           r['gt']['r2_ext'],
        'GT_k':             r['gt']['k'],
        'GT_k_prime':       r['gt']['kp'],
        'GT_R0sq':          r['gt']['R0sq'],
        'GT_R0sq_prime':    r['gt']['R0sqp'],
        'GT_C1':            r['gt']['C1'],
        'GT_C2':            r['gt']['C2'],
        'GT_C3':            r['gt']['C3'],
        'GT_C4':            r['gt']['C4'],
        'GT_C5':            r['gt']['C5'],
        'GT_OVERALL_PASS':  r['gt']['PASS'],
        'permutation_p':    round(r['perm_p'], 4),
    })
met_df = pd.DataFrame(metrics_rows)
p3 = os.path.join(OUTPUT_DIR, 'final_robust_metrics.csv')
met_df.to_csv(p3, index=False); print(f"  {p3}")

p4 = os.path.join(OUTPUT_DIR, 'final_robust_metrics.xlsx')
try:
    with pd.ExcelWriter(p4, engine='openpyxl') as writer:
        met_df.to_excel(writer, sheet_name='Metrics', index=False)
        pred_df.to_excel(writer, sheet_name='LOO Predictions')
        ext_df.to_excel(writer, sheet_name='External Test')
    print(f"  {p4}")
except Exception as e:
    print(f"  Excel skipped: {e}")


# ══ FINAL SUMMARY ═════════════════════════════════════════════════════════════
banner("FINAL SUMMARY")
W = 22
print(f"\n  {'Metric':<{W}}  {TARGET_FO:>20}  {TARGET_HIX:>20}")
print(f"  {'-'*65}")
fo, hix = RESULTS[TARGET_FO], RESULTS[TARGET_HIX]
def fmt(v):
    if isinstance(v, bool): return '✓' if v else '✗'
    return str(v)
rows_summary = [
    ('R² (in-sample)',  fo['R2_in'],          hix['R2_in']),
    ('Q² (LOO) ←',      fo['loo']['Q2'],      hix['loo']['Q2']),
    ('CCC (Lin) ★',     fo['loo']['CCC'],     hix['loo']['CCC']),
    ('Pearson r',       fo['loo']['r'],       hix['loo']['r']),
    ('Spearman ρ',      fo['loo']['rho'],     hix['loo']['rho']),
    ('RMSE (log)',      fo['loo']['RMSE'],    hix['loo']['RMSE']),
    ('RMSE (×real)',    fo['loo']['RMSE_real'],hix['loo']['RMSE_real']),
    ('MAE',             fo['loo']['MAE'],     hix['loo']['MAE']),
    ('Bias',            fo['loo']['Bias'],    hix['loo']['Bias']),
    ('Permutation p',   fo['perm_p'],         hix['perm_p']),
    ('External r²',     fo['gt']['r2_ext'],   hix['gt']['r2_ext']),
    ('GT k',            fo['gt']['k'],        hix['gt']['k']),
    ("GT k'",           fo['gt']['kp'],       hix['gt']['kp']),
    ('GT C1 Q²>0.5',    fo['gt']['C1'],       hix['gt']['C1']),
    ('GT C2 r²>0.6',    fo['gt']['C2'],       hix['gt']['C2']),
    ('GT C3 origin',    fo['gt']['C3'],       hix['gt']['C3']),
    ('GT C4 slopes',    fo['gt']['C4'],       hix['gt']['C4']),
    ('GT C5 R²₀ diff',  fo['gt']['C5'],       hix['gt']['C5']),
    ('GT OVERALL',      fo['gt']['PASS'],     hix['gt']['PASS']),
]
for label, fo_v, hix_v in rows_summary:
    print(f"  {label:<{W}}  {fmt(fo_v):>20}  {fmt(hix_v):>20}")

n_pass_fo  = sum([fo['gt'][f'C{i}']  for i in range(1,6)])
n_pass_hix = sum([hix['gt'][f'C{i}'] for i in range(1,6)])

print(f"""Outputs saved to: {os.path.abspath(OUTPUT_DIR)}
""")


## Model Interpretation SHAP (global, local, interaction) · LIME · PDP · ICE · Permutation Importance · Williams Plot · Residual Diagnostics The Setup cell uses variables already in memory: X_train, X_test, log_yfo_tr, log_yfo_te, bc_yfo_tr, lam_fo, feats_model, MATRIX_IDX, FIBER_IDX, add_interactions, bc_to_log, gpr_final, kernel, RESULTS, n_train. If you need the variables for hixson_rate, change TARGET = TARGET_FO to TARGET = TARGET_HIX in the setup cell and re-run.

In [ ]:
pip install shap lime -q

Setup —

---


Uses model objects already created.

In [ ]:

# ── Imports ───────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import copy
import warnings
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.patches import Patch

import shap
import lime
import lime.lime_tabular
from scipy.stats import pearsonr, spearmanr
from sklearn.preprocessing import StandardScaler
from sklearn.gaussian_process import GaussianProcessRegressor

warnings.filterwarnings('ignore')
shap.initjs()
# ─── Universal figure style (applies to every figure in this notebook) ───────
# Single source of truth for colours and matplotlib defaults.
MATRIX_BLUE  = '#185FA5'
FIBRE_GREEN  = '#3B6D11'
NEUTRAL_GREY = '#7f7f7f'
ACCENT_RED   = '#E24B4A'
ACCENT_AMBER = '#FEF3C7'

# Feature-class lookup: matrix-derived → blue, fibre-derived → green,
# composite / interfacial / mixed → grey. Used by every cell that colours
# bars or curves by feature identity.
FEATURE_CLASS = {
    # matrix
    'logP_matrix':                              'matrix',
    'matrix_CED':                               'matrix',
    'Half life_matrix_avg':                     'matrix',
    'matrix_sol_parameter':                     'matrix',
    'matrix_hansen_sol_parameter':              'matrix',
    'logKOC_matrix_avg':                        'matrix',
    # fibre
    'Hemicellulose_Percentage':                 'fibre',
    'Cellulose_Percentage':                     'fibre',
    'Lignin_Percentage':                        'fibre',
    'fiber_sol_parameter':                      'fibre',
    'fiber_dia_or_average_particle_size_mm':    'fibre',
    'fiber_CED':                                'fibre',
    'logP_fiber':                               'fibre',
    'Half life_fiber_avg':                      'fibre',
    'logKOC_fiber_avg':                         'fibre',
    'fiber_rotatable_bond_count':               'fibre',
    # composite / interfacial
    'tortuosity':                               'composite',
    'interface_rot':                            'composite',
    'Specific Interface Area':                  'composite',
    'sol_mismatch':                             'composite',
    'hydrophyllic_gradient':                    'composite',
    'flory_huggins':                            'composite',
}

def feat_color(name):
    cls = FEATURE_CLASS.get(name, 'composite')
    if cls == 'matrix':  return MATRIX_BLUE
    if cls == 'fibre':   return FIBRE_GREEN
    return NEUTRAL_GREY

# Matrix descriptor set (used by some legacy cells)
matrix_set = {k for k, v in FEATURE_CLASS.items() if v == 'matrix'}
fibre_set  = {k for k, v in FEATURE_CLASS.items() if v == 'fibre'}

plt.rcParams.update({
    'figure.dpi':         300,
    'font.family':        'DejaVu Sans',
    'figure.facecolor':   'white',
    'axes.facecolor':     'white',
    'savefig.facecolor':  'white',
    'axes.edgecolor':     'black',
    'axes.labelcolor':    'black',
    'axes.titlecolor':    'black',
    'xtick.color':        'black',
    'ytick.color':        'black',
    'text.color':         'black',
    'axes.grid':          False,
    'axes.spines.top':    False,
    'axes.spines.right':  False,
    'axes.spines.left':   True,
    'axes.spines.bottom': True,
    'axes.linewidth':     1.0,
    'axes.labelsize':     18,
    'axes.titlesize':     12,
    'xtick.labelsize':    18,
    'ytick.labelsize':    18,
    'legend.frameon':     True,
    'legend.edgecolor':   'black',
    'legend.facecolor':   'white',
})

def clean_axes(ax):
    """Apply post-plot clean-up: kill grid, force solid-black spines + text."""
    ax.grid(False)
    ax.set_facecolor('white')
    for s in ('top', 'right'):
        ax.spines[s].set_visible(False)
    for s in ('left', 'bottom'):
        ax.spines[s].set_color('black')
        ax.spines[s].set_linewidth(1.0)
    ax.tick_params(colors='black', which='both')
    ax.xaxis.label.set_color('black')
    ax.yaxis.label.set_color('black')
    if ax.get_title():
        ax.title.set_color('black')
# ─── end style block ─────────────────────────────────────────────────────────


# (rcParams set in style block below)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# ── Choose which target to interpret ──────────────────────────────────────────
# Change to TARGET_HIX to interpret the hixson model instead.
TARGET    = TARGET_HIX    # or TARGET_HIX
log_y_tr  = log_yfo_tr  if TARGET == TARGET_FO else log_yhix_tr
log_y_te  = log_yfo_te  if TARGET == TARGET_FO else log_yhix_te
bc_y_tr   = bc_yfo_tr   if TARGET == TARGET_FO else bc_yhix_tr
lam       = lam_fo      if TARGET == TARGET_FO else lam_hix
loo_preds = RESULTS[TARGET]['preds_loo']

print(f"Target selected: {TARGET}")
print(f"  Training: {n_train} samples × {X_train.shape[1]} base features")
print(f"  Box-Cox λ: {lam:.4f}")
print(f"  LOO Q² = {RESULTS[TARGET]['loo']['Q2']}")
print(f"  CCC    = {RESULTS[TARGET]['loo']['CCC']}")

# ── Re-fit GPR on full training set with interactions (for in-sample explainers) ─
sc_full = StandardScaler()
X_aug_tr = add_interactions(X_train)
X_aug_tr_s = sc_full.fit_transform(X_aug_tr)

gpr_explain = copy.deepcopy(gpr_final)
gpr_explain.fit(X_aug_tr_s, bc_y_tr)
print(f"\nGPR fitted. Kernel: {gpr_explain.kernel_}")

# ── Wrapper: raw features (8 columns) → log-scale prediction ─────────────────
def predict_log(X_raw):
    """For SHAP/LIME/permutation. Input: raw (unscaled) X with 8 columns.
       Output: back-transformed log(rate) prediction."""
    X_raw = np.atleast_2d(X_raw)
    X_a = add_interactions(X_raw)
    pred_bc = gpr_explain.predict(sc_full.transform(X_a))
    return bc_to_log(pred_bc, lam)

# ── SHAP background for KernelExplainer ───────────────────────────────────────
shap_background = shap.kmeans(X_train, 15)
explainer = shap.KernelExplainer(predict_log, shap_background)

# ── Pre-compute residuals for downstream cells ────────────────────────────────
residuals = log_y_tr - loo_preds
std_res   = residuals / residuals.std()

print("Setup complete. Run cells A–J in order or independently.")

### A. SHAP — Global Feature Importance


In [ ]:
# =============================================================================
# SHAP global feature importance + beeswarm (figure used in the manuscript shown later)
# =============================================================================
# =============================================================================

print("Computing SHAP values for all training samples ...")
shap_values = explainer.shap_values(X_train, nsamples=150)
shap_values = np.array(shap_values)

import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.patches import Patch
from matplotlib.gridspec import GridSpec

# ── In-cell font sizes (override global rcParams for this figure only) ───────
TICK_FS   = 10    # tick labels (feature names + x-axis numbers)
LABEL_FS  = 11    # axis labels ("Mean |SHAP value|" etc.)
VALUE_FS  = 9     # numerical labels at bar tips
LEGEND_FS = 10
CBAR_FS   = 9

# ── Figure layout via GridSpec: bar | beeswarm | colorbar ────────────────────
# Width ratios chosen so the bar chart and beeswarm get equal data space and
# the colorbar gets a thin dedicated column.
fig = plt.figure(figsize=(15, 6))
fig.patch.set_facecolor('white')
gs = GridSpec(1, 3, width_ratios=[1.0, 1.1, 0.04],
              wspace=0.40, figure=fig)
ax0 = fig.add_subplot(gs[0, 0])
ax1 = fig.add_subplot(gs[0, 1])
cax = fig.add_subplot(gs[0, 2])

# Leave room at the bottom for the legend
fig.subplots_adjust(bottom=0.18)

# ── Compute feature order (ascending — largest at top of barh) ───────────────
mean_abs = np.abs(shap_values).mean(0)
order_asc = np.argsort(mean_abs)
feat_names_sorted = [feats_model[i] for i in order_asc]
matrix_set = set([feats_model[i] for i in MATRIX_IDX])
colors_bar = ['#185FA5' if f in matrix_set else '#3B6D11'
              for f in feat_names_sorted]

# ── LEFT panel: mean |SHAP| bar chart ────────────────────────────────────────
y_pos = np.arange(len(order_asc))
bars = ax0.barh(y_pos, mean_abs[order_asc], color=colors_bar,
                height=0.65, edgecolor='black', linewidth=0.6)

# Value labels at bar tips
xmax = mean_abs.max()
for bar, v in zip(bars, mean_abs[order_asc]):
    ax0.text(v + xmax * 0.015, bar.get_y() + bar.get_height() / 2,
             f'{v:.3f}', va='center', ha='left',
             fontsize=VALUE_FS, color='black')

ax0.set_yticks(y_pos)
ax0.set_yticklabels(feat_names_sorted, fontsize=TICK_FS)
ax0.set_xlabel(r'Mean $|$SHAP value$|$  (impact on log rate)',
               fontsize=LABEL_FS)
ax0.set_xlim(0, xmax * 1.20)
ax0.tick_params(axis='x', labelsize=TICK_FS)

# Clean axis style
ax0.grid(False); ax0.set_facecolor('white')
for s in ('top', 'right'): ax0.spines[s].set_visible(False)
for s in ('left', 'bottom'):
    ax0.spines[s].set_color('black'); ax0.spines[s].set_linewidth(1.0)
ax0.tick_params(colors='black')

# ── RIGHT panel: beeswarm with density-based y-jitter ────────────────────────
def _density_jitter(values, max_jitter=0.38, n_bins=20):
    """Spread points wider in dense regions, narrower in sparse regions."""
    values = np.asarray(values)
    if len(values) == 0:
        return values
    edges = np.linspace(values.min(), values.max() + 1e-9, n_bins + 1)
    bin_idx = np.clip(np.digitize(values, edges) - 1, 0, n_bins - 1)
    bin_counts = np.bincount(bin_idx, minlength=n_bins)
    offsets = np.zeros(len(values))
    for b in range(n_bins):
        in_bin = np.where(bin_idx == b)[0]
        n_in = len(in_bin)
        if n_in == 0: continue
        density = n_in / max(bin_counts.max(), 1)
        local_jitter = max_jitter * np.sqrt(density)
        if n_in == 1:
            offsets[in_bin] = 0.0
        else:
            offsets[in_bin] = np.linspace(-local_jitter, local_jitter, n_in)
            rng = np.random.RandomState(b * 17 + 3)
            offsets[in_bin] = rng.permutation(offsets[in_bin])
    return offsets

for rank, fi in enumerate(order_asc):
    sv = shap_values[:, fi]
    fv = X_train[:, fi]
    p5, p95 = np.percentile(fv, [5, 95])
    if p95 - p5 > 1e-9:
        fv_norm = np.clip((fv - p5) / (p95 - p5), 0, 1)
    else:
        fv_norm = np.full_like(fv, 0.5)
    point_colors = cm.RdBu_r(fv_norm)
    y_offset = _density_jitter(sv, max_jitter=0.38)
    ax1.scatter(sv, np.full(len(sv), rank) + y_offset,
                c=point_colors, s=32, alpha=0.85,
                linewidths=0.3, edgecolors='black', zorder=3)

ax1.set_yticks(y_pos)
ax1.set_yticklabels(feat_names_sorted, fontsize=TICK_FS)
ax1.axvline(0, color='black', lw=0.9, ls='--', zorder=2)
ax1.set_xlabel(r'SHAP value  (impact on log rate)', fontsize=LABEL_FS)
ax1.tick_params(axis='x', labelsize=TICK_FS)

# Clean axis style
ax1.grid(False); ax1.set_facecolor('white')
for s in ('top', 'right'): ax1.spines[s].set_visible(False)
for s in ('left', 'bottom'):
    ax1.spines[s].set_color('black'); ax1.spines[s].set_linewidth(1.0)
ax1.tick_params(colors='black')

# ── Colorbar in its own column ───────────────────────────────────────────────
sm = cm.ScalarMappable(cmap='RdBu_r', norm=Normalize(vmin=0, vmax=1))
sm.set_array([])
cbar = fig.colorbar(sm, cax=cax)
cbar.set_label('Feature value  (low \u2192 high)', fontsize=CBAR_FS,
               rotation=270, labelpad=18)
cbar.set_ticks([0.0, 0.5, 1.0])
cbar.set_ticklabels(['low', 'median', 'high'])
cbar.ax.tick_params(labelsize=CBAR_FS)
cbar.outline.set_edgecolor('black')
cbar.outline.set_linewidth(0.6)

# ── Shared legend BELOW the figure (no collision with bars) ──────────────────
legend_handles = [
    Patch(facecolor='#185FA5', edgecolor='black', label='Matrix-derived'),
    Patch(facecolor='#3B6D11', edgecolor='black', label='Fibre-derived'),
]
fig.legend(handles=legend_handles, loc='lower center',
           bbox_to_anchor=(0.5, 0.02), ncol=2,
           frameon=True, edgecolor='black', fontsize=LEGEND_FS)

plt.savefig(f'shap_global_{TARGET}.png', bbox_inches='tight',
            dpi=300, facecolor='white')
plt.show()

print("\nTop 3 features by mean |SHAP|:")
for rank, fi in enumerate(order_asc[::-1][:3], 1):
    print(f"  {rank}. {feats_model[fi]}: mean |SHAP| = {mean_abs[fi]:.4f}")

### B. SHAP — Local Explanations (Waterfall Plots)
Explains 3 specific samples: best predicted, worst predicted, median residual.

In [ ]:
abs_res = np.abs(residuals)
idx_best   = int(np.argmin(abs_res))
idx_worst  = int(np.argmax(abs_res))
idx_median = int(np.argsort(abs_res)[len(abs_res)//2])

samples = {
    f'Best predicted (sample {idx_best}, residual={residuals[idx_best]:+.3f})':   idx_best,
    f'Worst predicted (sample {idx_worst}, residual={residuals[idx_worst]:+.3f})': idx_worst,
    f'Median (sample {idx_median}, residual={residuals[idx_median]:+.3f})':       idx_median,
}

# Increase figure size for better readability
fig, axes = plt.subplots(1, 3, figsize=(24, 8)) # Increased from (17, 5) to (24, 8)
for ax, (label, idx) in zip(axes, samples.items()):
    sv   = shap_values[idx]
    base = float(explainer.expected_value)
    pred = base + sv.sum()
    order_local = np.argsort(np.abs(sv))[::-1]

    running = base
    for rank, fi in enumerate(order_local):
        contrib = sv[fi]
        color = '#185FA5' if contrib > 0 else '#E24B4A'
        ax.barh(rank, contrib, left=running, color=color, height=0.6, alpha=0.85)
        # Modified text: removed truncation, increased fontsize
        ax.text(running + contrib/2, rank,
                f'{feats_model[fi]}\n({X_train[idx, fi]:.2f})', # Removed [:18] truncation
                ha='center', va='center', fontsize=18, color='black', fontweight='500') # Increased fontsize to 9

        running += contrib

    ax.axvline(base, color='black', lw=0.8, ls='--',
               label=f'Baseline: {base:.2f}')
    ax.axvline(pred, color='black', lw=1.0, ls='-',
               label=f'Predicted: {pred:.2f}')
    ax.set_xlabel('log rate')
    ax.set_title(label, fontsize=18) # Slightly increased title fontsize for consistency
    ax.set_yticks([]) # Keep empty y-ticks as feature names are inside bars
    ax.legend(fontsize=12, loc='upper right') # Changed legend location to upper left

plt.suptitle(f'SHAP waterfall — local explanations ({TARGET})',
             fontsize=14, fontweight='500') # Increased suptitle fontsize
plt.tight_layout()
plt.savefig(f'shap_local_{TARGET}.png', bbox_inches='tight', dpi=300)
plt.show()

### C. SHAP Dependence Plots
Shows how each top feature's SHAP value varies with its value, coloured by the strongest interacting feature.

In [ ]:
top4 = np.argsort(np.abs(shap_values).mean(0))[::-1][:4]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for plot_i, fi in enumerate(top4):
    sv_feat = shap_values[:, fi]
    fv_feat = X_train[:, fi]

    # Find feature whose value most strongly correlates with this feature's SHAP
    best_j, best_r = fi, 0
    for j in range(X_train.shape[1]):
        if j == fi: continue
        r = abs(np.corrcoef(sv_feat, X_train[:, j])[0, 1])
        if r > best_r: best_r, best_j = r, j

    interact_vals = X_train[:, best_j]
    norm = Normalize(vmin=np.percentile(interact_vals, 5),
                     vmax=np.percentile(interact_vals, 95))
    colors_dep = cm.RdYlGn(norm(interact_vals))

    axes[plot_i].scatter(fv_feat, sv_feat, c=colors_dep, s=30, alpha=0.8)
    axes[plot_i].axhline(0, color='black', lw=0.8, ls='--')
    axes[plot_i].set_xlabel(feats_model[fi], fontsize=18)
    axes[plot_i].set_ylabel('SHAP value', fontsize=18)
    axes[plot_i].set_title(
        f'{feats_model[fi]}\ncolour = {feats_model[best_j][:25]}', fontsize=12)
    sm = plt.cm.ScalarMappable(cmap='RdYlGn', norm=norm)
    sm.set_array([])
    plt.colorbar(sm, ax=axes[plot_i], shrink=0.7, pad=0.02)

plt.suptitle(f'SHAP dependence — top 4 features ({TARGET})',
             fontsize=12, fontweight='500')
plt.tight_layout()
plt.savefig(f'shap_dependence_{TARGET}.png', bbox_inches='tight', dpi=300)
plt.show()

### D. LIME — Local Linear Explanations
Fits a local linear model around each prediction. Useful as a non-ML-audience-friendly cross-check of SHAP local explanations.

In [ ]:

lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    X_train, feature_names=feats_model, mode='regression',
    discretize_continuous=True, random_state=RANDOM_SEED)

explain_indices = [idx_best, idx_worst, idx_median]
explain_labels  = ['Best predicted', 'Worst predicted', 'Median']

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
for ax, idx, label in zip(axes, explain_indices, explain_labels):
    exp = lime_explainer.explain_instance(
        X_train[idx], predict_log, num_features=8, num_samples=1000)
    items = exp.as_list()
    names  = [x[0] for x in items]
    vals   = [x[1] for x in items]
    colors = ['#185FA5' if v > 0 else '#E24B4A' for v in vals]

    ax.barh(range(len(vals)), vals, color=colors, height=0.6)
    ax.set_yticks(range(len(names)))
    ax.set_yticklabels(names, fontsize=18)
    ax.axvline(0, color='gray', lw=0.6)
    ax.set_xlabel('LIME contribution', fontsize=18)
    ax.set_title(f'{label}\nActual={log_y_tr[idx]:.2f}  Pred={loo_preds[idx]:.2f}',
                 fontsize=12)

plt.suptitle(f'LIME local explanations ({TARGET}) — '
             'blue = ↑ rate, red = ↓ rate', fontsize=11, fontweight='500')
plt.tight_layout()
plt.savefig(f'lime_local_{TARGET}.png', bbox_inches='tight', dpi=300)
plt.show()
print("Tip: cross-check with SHAP waterfall (Cell B) — agreement validates the explanation.")


### E. Partial Dependence Plots (PDP)
Marginal effect of each feature, averaged across all training samples. Shows population-average response.

In [ ]:

n_grid = 40
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

avg_pred_baseline = np.mean([predict_log(X_train[i:i+1]).item() for i in range(n_train)])
matrix_set = set([feats_model[i] for i in MATRIX_IDX])

for fi, (fname, ax) in enumerate(zip(feats_model, axes)):
    x_grid = np.linspace(X_train[:, fi].min(), X_train[:, fi].max(), n_grid)
    pdp_vals = np.zeros(n_grid)
    for g, x_val in enumerate(x_grid):
        X_p = X_train.copy(); X_p[:, fi] = x_val
        pdp_vals[g] = predict_log(X_p).mean()

    is_matrix = fname in matrix_set
    color = '#185FA5' if is_matrix else '#3B6D11'

    ax.plot(x_grid, pdp_vals, color=color, lw=2)
    ax.fill_between(x_grid, pdp_vals, alpha=0.12, color=color)
    ax.plot(X_train[:, fi], np.full(n_train, pdp_vals.min() - 0.02),
            '|', color='gray', alpha=0.4, ms=5)
    ax.axhline(avg_pred_baseline, color='black', lw=0.8, ls='--')
    ax.set_xlabel(fname, fontsize=8)
    if fi % 4 == 0: ax.set_ylabel('Avg log rate', fontsize=9)
    tag = 'matrix' if is_matrix else 'fiber'
    ax.set_title(f'{fname[:30]}\n[{tag}]', fontsize=8)

plt.suptitle(f'Partial Dependence Plots ({TARGET})\n'
             'Blue = matrix | Green = fiber',
             fontsize=11, fontweight='500')
plt.tight_layout()
plt.savefig(f'pdp_{TARGET}.png', bbox_inches='tight', dpi=300)
plt.show()
print("Steeper slope = stronger marginal effect. Non-linear curves = GPR captured non-linearity.")


### F. Individual Conditional Expectation (ICE) Plots
Like PDP but shows every sample's individual trajectory. Crossing lines reveal interactions.

In [ ]:

top2 = np.argsort(np.abs(shap_values).mean(0))[::-1][:2]
n_grid_ice = 30

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, fi in zip(axes, top2):
    fname = feats_model[fi]
    x_grid = np.linspace(X_train[:, fi].min(), X_train[:, fi].max(), n_grid_ice)
    ice = np.zeros((n_train, n_grid_ice))
    for g, x_val in enumerate(x_grid):
        X_ice = X_train.copy(); X_ice[:, fi] = x_val
        ice[:, g] = predict_log(X_ice)

    norm = Normalize(vmin=X_train[:, fi].min(), vmax=X_train[:, fi].max())
    for i in range(n_train):
        c = cm.RdBu_r(norm(X_train[i, fi]))
        ax.plot(x_grid, ice[i], color=c, alpha=0.3, lw=0.8)

    ax.plot(x_grid, ice.mean(0), color='black', lw=2.5,
            label='PDP (mean ICE)', zorder=5)
    ax.plot(X_train[:, fi], np.full(n_train, ice.min() - 0.03),
            '|', color='gray', alpha=0.5, ms=5)
    ax.set_xlabel(fname, fontsize=18)
    ax.set_ylabel('log rate', fontsize=18)
    ax.set_title(f'ICE — {fname}\ncolour = actual feature value', fontsize=18, fontweight='normal')
    ax.legend(fontsize=12)
    sm = plt.cm.ScalarMappable(cmap='RdBu_r', norm=norm); sm.set_array([])
    plt.colorbar(sm, ax=ax, label=fname, shrink=0.7)

plt.suptitle(f'ICE plots ({TARGET}) — crossing lines = interactions',
             fontsize=11, fontweight='500')
plt.tight_layout()
plt.savefig(f'ice_{TARGET}.png', bbox_inches='tight', dpi=300)
plt.show()


#### F.1. ICE Slope

In [ ]:
top2 = np.argsort(np.abs(shap_values).mean(0))[::-1][:2]

n_grid_ice = 30

xmin, xmax = -2, 3

for fi in top2:

    fname = feats_model[fi]

    # grid for current feature
    x_grid = np.linspace(
        X_train[:, fi].min(),
        X_train[:, fi].max(),
        n_grid_ice
    )

    # ICE matrix
    ice = np.zeros((n_train, n_grid_ice))

    for g, x_val in enumerate(x_grid):

        X_ice = X_train.copy()
        X_ice[:, fi] = x_val

        ice[:, g] = predict_log(X_ice)

    # first derivative
    dice_dx = np.gradient(ice, x_grid, axis=1)

    # selected x-range
    mask = (x_grid >= xmin) & (x_grid <= xmax)

    # average derivative
    mean_slope = dice_dx[:, mask].mean()

    # convert log-slope to fold-change
    # use np.exp for natural log
    # use 10**mean_slope for log10
    fold_change = np.exp(mean_slope)

    print("\n----------------------------")
    print(f"Feature: {fname}")

    if mean_slope < 0:

        reduction = 1 / fold_change

        print(
            f"Approximately {abs(mean_slope):.2f} log-units "
            f"per unit increase in {fname}, "
            f"equivalent to a ~{reduction:.1f}-fold reduction "
            f"in predicted rate."
        )

    else:

        print(
            f"Approximately {mean_slope:.2f} log-units "
            f"per unit increase in {fname}, "
            f"equivalent to a ~{fold_change:.1f}-fold increase "
            f"in predicted rate."
        )

### G. Actual vs Predicted Plot
LOO predictions on training + the 9 external test samples on one figure with residual diagnostics.

In [ ]:
ext_preds = predict_log(X_test)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
fig.patch.set_facecolor('white')

# ── Left: LOO + external test ─────────────────────────────────────────────────
ax0 = axes[0]
loo_q2  = RESULTS[TARGET]['loo']['Q2']
loo_ccc = RESULTS[TARGET]['loo']['CCC']
ext_q2  = RESULTS[TARGET]['gt']['r2_ext']

ax0.scatter(log_y_tr, loo_preds, c='#185FA5', alpha=0.8, s=55,
            edgecolors='black', linewidths=0.4,
            label=f'LOO train (n = {n_train})')
ax0.scatter(log_y_te, ext_preds, c='#E24B4A', marker='D', s=75, alpha=0.95,
            edgecolors='black', linewidths=0.6,
            label=f'External test (n = {X_test.shape[0]})')

lims = [min(log_y_tr.min(), log_y_te.min(), loo_preds.min(), ext_preds.min()),
        max(log_y_tr.max(), log_y_te.max(), loo_preds.max(), ext_preds.max())]
margin = (lims[1]-lims[0])*0.05; lims = [lims[0]-margin, lims[1]+margin]
ax0.plot(lims, lims, 'k--', lw=1.0, alpha=0.6, label='y = x')
ax0.set_xlim(lims); ax0.set_ylim(lims)
ax0.set_xlabel(f'Actual log({TARGET})')
ax0.set_ylabel(f'Predicted log({TARGET})')
ax0.set_aspect('equal')

# Metrics box: clean, top-left, readable size
metrics_text_left = (
    f'$Q^{{2}}_{{\\rm LOO}}$  = {loo_q2}\n'
    f'$CCC_{{\\rm LOO}}$ = {loo_ccc}\n'
    f'$r^{{2}}_{{\\rm ext}}$    = {ext_q2}'
)
ax0.text(0.04, 0.96, metrics_text_left, transform=ax0.transAxes,
         fontsize=12, family='serif', va='top', ha='left',
         bbox=dict(boxstyle='round,pad=0.45', facecolor='white',
                   edgecolor='black', linewidth=0.6))

ax0.legend(loc='lower right', frameon=True, edgecolor='black')

# Clean axis style
ax0.grid(False); ax0.set_facecolor('white')
for s in ('top','right'): ax0.spines[s].set_visible(False)
for s in ('left','bottom'):
    ax0.spines[s].set_color('black'); ax0.spines[s].set_linewidth(1.0)
ax0.tick_params(colors='black', which='both')

# ── Right: residuals vs predicted ─────────────────────────────────────────────
ax1 = axes[1]
sigma = residuals.std()

# Shaded ±2σ band (clearer than two dotted lines + small label)
ax1.axhspan(-2*sigma, 2*sigma, color='#7f7f7f', alpha=0.10,
            label=r'$\pm 2\sigma$ band')
ax1.axhline(0, color='black', lw=0.9, ls='--')

ax1.scatter(loo_preds, residuals, c='#185FA5', alpha=0.8, s=55,
            edgecolors='black', linewidths=0.4)

# Larger annotations for outliers
for i, (p, r) in enumerate(zip(loo_preds, residuals)):
    if abs(r) > 1.85*sigma:
        ax1.annotate(f'[{i}]', (p, r),
                     xytext=(7, 6), textcoords='offset points',
                     color='#E24B4A', fontweight='bold')

ax1.set_xlabel(f'Predicted log({TARGET})')
ax1.set_ylabel('Residual (actual − predicted)')
ax1.legend(loc='upper right', frameon=True, edgecolor='black')

# Residual stats summary in top-left (cleaner than two-line title)
stats_text = (
    f'$\\bar{{r}}$       = {residuals.mean():+.3f}\n'
    f'$\\sigma_{{r}}$    = {residuals.std():.3f}\n'
    f'$|r| > 2\\sigma$  : {(np.abs(residuals) > 2*sigma).sum()} pts'
)
ax1.text(0.04, 0.96, stats_text, transform=ax1.transAxes,
         fontsize=12, family='serif', va='top', ha='left',
         bbox=dict(boxstyle='round,pad=0.45', facecolor='white',
                   edgecolor='black', linewidth=0.6))

# Clean axis style
ax1.grid(False); ax1.set_facecolor('white')
for s in ('top','right'): ax1.spines[s].set_visible(False)
for s in ('left','bottom'):
    ax1.spines[s].set_color('black'); ax1.spines[s].set_linewidth(1.0)
ax1.tick_params(colors='black', which='both')

plt.tight_layout()
plt.savefig(f'actual_vs_predicted_{TARGET}.png', bbox_inches='tight',
            dpi=300, facecolor='white')
plt.show()

print(f"\nResidual statistics:")
print(f"  Mean   = {residuals.mean():+.4f}  (should be ~ 0)")
print(f"  Std    = {residuals.std():.4f}")
print(f"  Range  = [{residuals.min():+.3f}, {residuals.max():+.3f}]")
print(f"  Outside +/-2 sigma: {(np.abs(residuals) > 2*sigma).sum()} samples")

### H. Williams Plot — Applicability Domain
Identifies high-leverage samples and outliers. Defines the prediction reliability domain.

In [ ]:
# =============================================================================
# Applicability domain — Williams plot (leverage)
# =============================================================================
#

import warnings
warnings.filterwarnings('ignore')

import copy
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneOut
from sklearn.gaussian_process import GaussianProcessRegressor

from colored_analyses_b2_b6 import (
    reproduce_pipeline, make_add_interactions, default_gpr_factory,
    MATRIX_BLUE, FIBRE_GREEN,
)

# ---- Configuration ----------------------------------------------------------
DATASET_PATH = 'dataset.xlsx'
TARGET       = 'hixson_rate'        # change to 'first_order_rate' if desired
OUT_PNG      = f'williams_plot_{TARGET}.png'
OUT_PDF      = f'williams_plot_{TARGET}.pdf'

# Shading colours
IN_DOMAIN_GREEN  = '#E3F0DA'        # pale green  = inside applicability domain
OUT_DOMAIN_BLUE  = '#DCE6F2'        # pale blue   = outside applicability domain

# Marker colours
TRAIN_COLOR = '#185FA5'
EXT_COLOR   = '#3B6D11'
LINE_RED    = '#E24B4A'
TEXT_COLOR  = 'black'

# Typography — large, for an unfamiliar reader
TICK_FS    = 15
LABEL_FS   = 18
LEGEND_FS  = 14
ANNOT_FS   = 14
TITLE_FS   = 19
SUBTITLE_FS = 13
PCT_FS     = 15

# ============================================================================
# 1. Pipeline — base-descriptor matrices + target
# ============================================================================
print(f"[setup] Reproducing pipeline ({TARGET}) ...")
pipe = reproduce_pipeline(DATASET_PATH, target=TARGET, verbose=False)
X_tr_base  = np.asarray(pipe['X_train_base'], float)
X_ext_base = np.asarray(pipe['X_ext_base'],   float)
y_tr_bc    = np.asarray(pipe['y_train_bc'],   float)
y_ext      = np.asarray(pipe['y_ext_log'],    float)
feat_names = list(pipe['feat_names'])

n_tr, p = X_tr_base.shape
n_ext   = X_ext_base.shape[0]
print(f"[setup] Training {X_tr_base.shape}, External {X_ext_base.shape}")

# ============================================================================
# 2. Leverage with the (p+1) intercept convention
# ============================================================================
sc = StandardScaler().fit(X_tr_base)
Xs_tr  = sc.transform(X_tr_base)
Xs_ext = sc.transform(X_ext_base)

# Design matrix WITH an intercept column -> p+1 columns
D_tr  = np.column_stack([np.ones(n_tr),  Xs_tr])     # n_tr  x (p+1)
D_ext = np.column_stack([np.ones(n_ext), Xs_ext])    # n_ext x (p+1)
p_eff = D_tr.shape[1]                                # = p + 1

# Training-only Gram matrix
DtD_inv = np.linalg.pinv(D_tr.T @ D_tr)

# Leverage = diagonal of the hat matrix
h_train = np.einsum('ij,jk,ik->i', D_tr,  DtD_inv, D_tr)
h_ext   = np.einsum('ij,jk,ik->i', D_ext, DtD_inv, D_ext)

# Standard QSAR threshold: h* = 3 * (p+1) / n
h_star = 3.0 * p_eff / n_tr

# --- correctness check: training leverage must lie in [0,1] -----------------
tol = 1e-8
assert h_train.min() >= -tol and h_train.max() <= 1.0 + tol, (
    f"Training leverage out of [0,1]: min={h_train.min():.4f}, "
    f"max={h_train.max():.4f}. Calculation is wrong — do not use this plot.")
print(f"[check] Training leverage in [0,1]: "
      f"min={h_train.min():.4f}, max={h_train.max():.4f}  OK")
print(f"[check] h* = 3(p+1)/n = 3*{p_eff}/{n_tr} = {h_star:.4f}")

# ============================================================================
# 3. Standardised residuals (LOO) for the vertical axis
# ============================================================================
add_int = make_add_interactions(feat_names)
sc_aug = StandardScaler().fit(add_int(X_tr_base))
Xs_aug_tr = sc_aug.transform(add_int(X_tr_base))

gpr_full = copy.deepcopy(default_gpr_factory())
gpr_full.fit(Xs_aug_tr, y_tr_bc)
fitted_kernel = gpr_full.kernel_

loo_pred = np.zeros(n_tr)
for tr, te in LeaveOneOut().split(Xs_aug_tr):
    g = GaussianProcessRegressor(kernel=fitted_kernel, alpha=gpr_full.alpha,
                                  normalize_y=True, optimizer=None)
    g.fit(Xs_aug_tr[tr], y_tr_bc[tr])
    loo_pred[te[0]] = g.predict(Xs_aug_tr[te])[0]
resid_tr = y_tr_bc - loo_pred
sd_resid = resid_tr.std()
std_resid_tr = resid_tr / sd_resid

Xs_aug_ext = sc_aug.transform(add_int(X_ext_base))
ext_pred = gpr_full.predict(Xs_aug_ext)
std_resid_ext = (y_ext - ext_pred) / sd_resid

# ============================================================================
# 4. In / out-of-domain verdicts and percentages
# ============================================================================
def in_domain(h, r):
    return (h <= h_star) & (np.abs(r) <= 3.0)

tr_in  = in_domain(h_train, std_resid_tr)
ext_in = in_domain(h_ext,   std_resid_ext)

pct_tr_in  = 100.0 * tr_in.sum()  / n_tr
pct_ext_in = 100.0 * ext_in.sum() / n_ext

# ============================================================================
# 5. Williams plot — shaded, large fonts, reader-friendly
# ============================================================================
fig, ax = plt.subplots(figsize=(12, 8))
fig.patch.set_facecolor('white')

# Axis ranges (leave headroom for shading + annotations)
x_max = max(h_train.max(), h_ext.max(), h_star) * 1.15
y_abs = max(4.0, np.abs(np.concatenate([std_resid_tr, std_resid_ext])).max() * 1.15)
ax.set_xlim(0, x_max)
ax.set_ylim(-y_abs, y_abs)

# --- Shading -----------------------------------------------------------------
# GREEN = inside the applicability domain: leverage <= h* AND |residual| <= 3
ax.add_patch(plt.Rectangle((0, -3), h_star, 6,
                            facecolor=IN_DOMAIN_GREEN, edgecolor='none', zorder=0))
# BLUE = outside the applicability domain: everywhere else
#   (a) right of h* (high leverage), full height
ax.add_patch(plt.Rectangle((h_star, -y_abs), x_max - h_star, 2 * y_abs,
                            facecolor=OUT_DOMAIN_BLUE, edgecolor='none', zorder=0))
#   (b) left of h* but beyond +/-3 sigma (residual outliers)
ax.add_patch(plt.Rectangle((0,  3), h_star, y_abs - 3,
                            facecolor=OUT_DOMAIN_BLUE, edgecolor='none', zorder=0))
ax.add_patch(plt.Rectangle((0, -y_abs), h_star, y_abs - 3,
                            facecolor=OUT_DOMAIN_BLUE, edgecolor='none', zorder=0))

# Zone labels inside the shaded areas
ax.text(h_star * 0.5, y_abs * 0.86, 'INSIDE\napplicability domain',
        ha='center', va='center', fontsize=PCT_FS, fontweight='bold',
        color='#3B6D11', zorder=2)
ax.text(h_star + (x_max - h_star) * 0.5, y_abs * 0.86,
        'OUTSIDE\napplicability domain',
        ha='center', va='center', fontsize=PCT_FS, fontweight='bold',
        color='#185FA5', zorder=2)

# --- Threshold lines ---------------------------------------------------------
ax.axvline(h_star, color='purple', ls='--', lw=2.0, zorder=2)
ax.axhline( 3, color=LINE_RED, ls='--', lw=1.6, zorder=2)
ax.axhline(-3, color=LINE_RED, ls='--', lw=1.6, zorder=2)
ax.axhline( 0, color='gray', ls=':', lw=1.0, zorder=1)

# h* label on the line
ax.text(h_star, -y_abs * 0.97, f'  $h^*$ = {h_star:.2f}',
        ha='left', va='bottom', fontsize=ANNOT_FS, fontweight='bold',
        color=LINE_RED, zorder=3)

# --- Data points -------------------------------------------------------------
ax.scatter(h_train, std_resid_tr, c=TRAIN_COLOR, s=95, alpha=0.85,
           edgecolors='black', linewidths=0.7, zorder=4)
ax.scatter(h_ext, std_resid_ext, c=EXT_COLOR, marker='D', s=135, alpha=0.95,
           edgecolors='black', linewidths=0.9, zorder=5)

# Label external composites that fall outside the domain
for i, (h, r) in enumerate(zip(h_ext, std_resid_ext)):
    if not ext_in[i]:
        ax.annotate(f'T{i}', (h, r), xytext=(9, 8),
                    textcoords='offset points', fontsize=ANNOT_FS,
                    fontweight='normal', color=TEXT_COLOR, zorder=6)

# --- Percentage call-outs ----------------------------------------------------
ax.text(0.03, 0.03,
        f'Training composites inside domain:  {tr_in.sum()}/{n_tr}  '
        f'({pct_tr_in:.0f}%)\n'
        f'External composites inside domain:  {ext_in.sum()}/{n_ext}  '
        f'({pct_ext_in:.0f}%)',
        transform=ax.transAxes, ha='left', va='bottom',
        fontsize=PCT_FS, fontweight='normal', color=TEXT_COLOR,
        bbox=dict(boxstyle='round,pad=0.5', facecolor='white',
                  edgecolor='black', linewidth=1.0), zorder=7)

# --- Axes, title, sub-title --------------------------------------------------
ax.set_xlabel('Leverage,  $h_i$   (distance from training data)',
              fontsize=LABEL_FS, color=TEXT_COLOR)
ax.set_ylabel('Standardised residual', fontsize=LABEL_FS, color=TEXT_COLOR)
#ax.set_title('Applicability domain — Williams plot',
             #fontsize=TITLE_FS, fontweight='bold', color=TEXT_COLOR, pad=34)
#ax.text(0.5, 1.015,
        #'Green zone: predictions reliable.  Blue zone: high leverage or large '
        #'residual — predictions are extrapolations.',
        #transform=ax.transAxes, ha='center', va='bottom',
        #fontsize=SUBTITLE_FS, style='italic', color='#444444')

# --- Legend ------------------------------------------------------------------
legend_handles = [
    Line2D([0],[0], marker='o', color='w', markerfacecolor=TRAIN_COLOR,
           markeredgecolor='black', markersize=13,
           label=f'Training composite (n = {n_tr})'),
    Line2D([0],[0], marker='D', color='w', markerfacecolor=EXT_COLOR,
           markeredgecolor='black', markersize=13,
           label=f'External test composite (n = {n_ext})'),
    Line2D([0],[0], color='purple', ls='--', lw=2.0,
           label=f'Leverage threshold  $h^*$ = {h_star:.2f}'),
    Line2D([0],[0], color=LINE_RED, ls='--', lw=2.0,
           label=r'Residual bounds  $\pm 3$'),
    Patch(facecolor=IN_DOMAIN_GREEN, edgecolor='black',
          label='Inside applicability domain'),
    Patch(facecolor=OUT_DOMAIN_BLUE, edgecolor='black',
          label='Outside applicability domain'),
]
ax.legend(handles=legend_handles, loc='lower right',
          frameon=True, edgecolor='black', fontsize=LEGEND_FS,
          framealpha=0.95)

ax.tick_params(colors=TEXT_COLOR, which='both', labelsize=TICK_FS)
for spn in ('top', 'right'): ax.spines[spn].set_visible(False)
for spn in ('left', 'bottom'):
    ax.spines[spn].set_color('black'); ax.spines[spn].set_linewidth(1.2)
for lbl in ax.get_xticklabels() + ax.get_yticklabels():
    lbl.set_color(TEXT_COLOR)

plt.tight_layout()
plt.savefig(OUT_PNG, dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig(OUT_PDF,            bbox_inches='tight', facecolor='white')
plt.show()
print(f"\n[output] Saved {OUT_PNG} and {OUT_PDF}")

# ============================================================================
# 6. Numerical summary
# ============================================================================
print("\n" + "=" * 70)
print(f"WILLIAMS-PLOT APPLICABILITY DOMAIN — {TARGET}")
print("=" * 70)
print(f"\n  Convention   : h* = 3(p+1)/n  (intercept included)")
print(f"  p+1          : {p_eff}   (8 base descriptors + intercept)")
print(f"  n (training) : {n_tr}")
print(f"  h*           : {h_star:.4f}")
print(f"  Training leverage range : [{h_train.min():.4f}, {h_train.max():.4f}]")
print(f"\n  Training composites inside domain : "
      f"{tr_in.sum()}/{n_tr}  ({pct_tr_in:.1f}%)")
print(f"  External composites inside domain : "
      f"{ext_in.sum()}/{n_ext}  ({pct_ext_in:.1f}%)")
print(f"\nExternal composites:")
print(f"  {'ID':<5s}{'leverage':>11s}{'std resid':>12s}{'  verdict':>14s}")
for i in range(n_ext):
    print(f"  T{i:<4d}{h_ext[i]:>11.4f}{std_resid_ext[i]:>12.4f}"
          f"{('inside' if ext_in[i] else 'OUTSIDE'):>14s}")
print("\nNote: leverage uses the (p+1) intercept convention on the 8 base")
print("descriptors. The fitted model also uses 15 interaction terms derived")
print("from these descriptors; the leverage domain is the base-descriptor")
print("applicability domain. State this in the manuscript Methods and report")
print("alongside the GPR predictive-uncertainty domain.")

### I. Residual-Descriptor Correlation Diagnostic
If any descriptor correlates with residuals (|r|>0.3, p<0.05), the model is under-using that feature or missing an interaction.

In [ ]:

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()
matrix_set = set([feats_model[i] for i in MATRIX_IDX])

print(f"  {'Feature':<46} {'r':>8}  {'ρ':>8}  {'p':>9}  flag")
print(f"  {'-'*82}")
for fi, (fname, ax) in enumerate(zip(feats_model, axes)):
    fv = X_train[:, fi]
    r,  p_r  = pearsonr(fv, residuals)
    rho, p_s = spearmanr(fv, residuals)
    flag = '← SYSTEMATIC' if abs(r) > 0.3 and p_r < 0.05 else ''
    print(f"  {fname:<46} {r:>+8.3f}  {rho:>+8.3f}  {p_r:>9.4f}  {flag}")

    color = '#185FA5' if fname in matrix_set else '#3B6D11'
    ax.scatter(fv, residuals, c=color, alpha=0.7, s=30)
    z = np.polyfit(fv, residuals, 1)
    x_line = np.linspace(fv.min(), fv.max(), 50)
    ax.plot(x_line, np.polyval(z, x_line), 'k--', lw=1.2, alpha=0.6)
    ax.axhline(0, color='gray', lw=0.6)
    ax.set_xlabel(fname, fontsize=12)
    if fi % 4 == 0: ax.set_ylabel('Residual', fontsize=9)
    ax.set_title(f'r={r:+.3f}  ρ={rho:+.3f}\np={p_r:.3f}', fontsize=9)
    if abs(r) > 0.3 and p_r < 0.05:
        ax.set_facecolor('#FEF3C7')

plt.suptitle(f'Residual vs feature ({TARGET}) — '
             'yellow = significant systematic bias',
             fontsize=11, fontweight='500')
plt.tight_layout()
plt.savefig(f'residual_diagnostics_{TARGET}.png', bbox_inches='tight', dpi=300)
plt.show()

print(f"\nIf any feature flagged: consider adding its squared term or an interaction.")
print(f"No flagged features = model has captured the systematic structure of all features.")


## Y-Scrambling

In [ ]:
# ══ STEP 9 — Y-Scrambling Visualization ═══════════════════════════════════════
banner("Step 9 — Y-Scrambling Visualization")
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
targets = [TARGET_FO, TARGET_HIX]
colors = ['#185FA5', '#E24B4A']
titles = ['First Order Rate', 'Hixson Rate']

for i, tname in enumerate(targets):
    # Extract data already computed in Step 7
    r = RESULTS[tname]
    null_dist = r['null_q2']
    obs_q2 = r['loo']['Q2']
    p_val = r['perm_p']

    # Plot histogram of scrambled models
    axes[i].hist(null_dist, bins=30, color='gray', alpha=0.6, edgecolor='black',
                 label=f'Scrambled Null Distribution\n(n={len(null_dist)})')

    # Plot the observed model performance
    axes[i].axvline(obs_q2, color=colors[i], linestyle='--', linewidth=3,
                    label=f'Actual Model ($Q^2$ = {obs_q2:.3f})\n$p$ = {p_val:.4f}')

    # Formatting
    #axes[i].set_title(f'Y-Scrambling Test: {titles[i]}', fontsize=14, fontweight='bold')
    axes[i].set_xlabel('Cross-Validated $Q^2$ Value', fontsize=18)
    axes[i].set_ylabel('Frequency', fontsize=18)

    # Add a safe zone / significance marker
    axes[i].axvspan(-1.0, np.percentile(null_dist, 95), color='red', alpha=0.05, label='95% Confidence Threshold')
    axes[i].legend(loc='upper right', fontsize=12, shadow=True)
    #axes[i].grid(False, alpha=0.3)
    axes[i].set_xlim([min(null_dist) - 0.2, obs_q2 + 0.1])

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'y_scrambling_histograms.png'), dpi=300, bbox_inches='tight')
print(f"  Saved Y-Scrambling plot to: {os.path.join(OUTPUT_DIR, 'y_scrambling_histograms.png')}")
plt.show()

## Feature Properties

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from scipy import stats

# Set plotting style
plt.style.use('seaborn-v0_8-white')
sns.set_context("paper", font_scale=1.5)
params = {'legend.fontsize': 12, 'axes.labelsize': 18, 'axes.titlesize': 16, 'figure.dpi': 300}
plt.rcParams.update(params)

# 1. Load your data
df = pd.read_excel('dataset.xlsx')

# Define your features and targets
features = ['logP_matrix', 'tortuosity', 'Hemicellulose_Percentage', 'matrix_CED',
            'Half life_matrix_avg', 'fiber_dia_or_average_particle_size_mm']
targets = ['first_order_rate', 'hixson_rate']

# ==========================================
# PART 1: FEATURE SUMMARIZATION (TABLE)
# ==========================================
def summarize_features(data, feature_list):
    summary = data[feature_list].describe().T
    summary = summary[['mean', 'std', 'min', '50%', 'max']]
    summary.columns = ['Mean', 'Std Dev', 'Min', 'Median', 'Max']
    print("--- Feature Summary Table ---")
    print(summary.round(3))
    summary.to_csv('feature_summary_table.csv')
    return summary

# ==========================================
# PART 2: TARGET DISTRIBUTION (BEFORE/AFTER)
# ==========================================
def plot_target_distribution(data, target_col):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Original Distribution
    sns.histplot(data[target_col], kde=True, ax=axes[0], color='skyblue')
    axes[0].set_title(f'Original {target_col}')

    # Box-Cox Transformed (Simulated to show the logic)
    transformed_data, _ = stats.boxcox(data[target_col] + 1e-6)
    sns.histplot(transformed_data, kde=True, ax=axes[1], color='salmon')
    axes[1].set_title(f'Box-Cox Transformed {target_col}')

    plt.tight_layout()
    plt.savefig(f'distribution_{target_col}.png')

# ==========================================
# PART 3: FEATURE CORRELATION HEATMAP
# ==========================================
def plot_correlation(data, feature_list):
    plt.figure(figsize=(10, 8))
    corr = data[feature_list].corr()
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, annot=True, cmap='RdBu_r', center=0, fmt='.2f', square=True)
    plt.title('Feature Correlation Matrix')
    plt.savefig('feature_correlation.png')

# ==========================================
# PART 4: CHEMICAL SPACE VISUALIZATION (PCA)
# ==========================================
def plot_chemical_space(data, feature_list, split_col=None):
    """
    split_col: column identifying 'Train', 'Test', or 'Outlier'
    """
    x = data[feature_list].values
    x_scaled = StandardScaler().fit_transform(x)

    pca = PCA(n_components=2)
    components = pca.fit_transform(x_scaled)
    var_exp = pca.explained_variance_ratio_

    plt.figure(figsize=(8, 6))

    if split_col and split_col in data.columns:
        # Define specific colors for your defense
        color_map = {'Train': '#185FA5', 'Test': '#3B6D11', 'Outlier': '#E24B4A'}
        sns.scatterplot(x=components[:, 0], y=components[:, 1],
                        hue=data[split_col], palette=color_map,
                        s=100, edgecolor='black', alpha=0.8)
    else:
        sns.scatterplot(x=components[:, 0], y=components[:, 1], s=100, alpha=0.7)

    plt.xlabel(f'Principal Component 1 ({var_exp[0]:.1%})')
    plt.ylabel(f'Principal Component 2 ({var_exp[1]:.1%})')
    plt.title('Chemical Space Coverage (PCA)')
    plt.legend(title='Dataset Split', loc='best')
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.savefig('chemical_space_pca.png')

# Execute all steps
summary_df = summarize_features(df, features)
for t in targets:
    plot_target_distribution(df, t)
plot_correlation(df, features)
# Note: Add a column 'Set' to your CSV identifying Train/Test/Outlier points first!
plot_chemical_space(df, features, split_col='Set')

In [ ]:
import copy, numpy as np
from sklearn.preprocessing import StandardScaler
from font_colored_analyses_b2_b6 import (run_all, LOG_TRANSFORM_FEATS, bc_to_log,
                            default_gpr_factory, make_add_interactions)

add_int = make_add_interactions(feats_model)
Xtr_int = add_int(X_train)
Xte_int = add_int(X_test)
sc = StandardScaler().fit(Xtr_int)
gpr = default_gpr_factory(n_restarts=10)
gpr.fit(sc.transform(Xtr_int), bc_y_tr)
ext_pred_bc = gpr.predict(sc.transform(Xte_int))
ext_pred_log = bc_to_log(ext_pred_bc, lam)

X_train_raw_for_b6 = X_train.copy()
for f in LOG_TRANSFORM_FEATS:
    if f in feats_model:
        ci = feats_model.index(f)
        X_train_raw_for_b6[:, ci] = np.exp(X_train_raw_for_b6[:, ci])

results = run_all(
    X_train_base=X_train, y_train_bc=bc_y_tr, y_train_log=log_y_tr,
    X_ext_base=X_test, y_ext_log=log_y_te,
    ext_pred_log=ext_pred_log,
    lam=lam, feat_names=feats_model,
    X_train_raw_for_b6=X_train_raw_for_b6,
    target_name='first_order_rate', out_dir='./analyses_output',
)

## Per Composite GPR improvement

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.preprocessing import StandardScaler
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C, WhiteKernel
from sklearn.model_selection import LeaveOneOut
from scipy.stats import boxcox
from scipy.spatial.distance import cdist

# ---- Configuration ----------------------------------------------------------
DATASET_PATH = 'dataset.xlsx'
OUT_PNG      = 'Figure_per_composite_improvement.png'
OUT_PDF      = 'Figure_per_composite_improvement.pdf'
RANDOM_STATE = 42

FIBRE_GREEN  = '#3B6D11'

# ============================================================================
# 1. Reproduce the pipeline
# ============================================================================
df_raw = pd.read_excel(DATASET_PATH)
df_raw = df_raw.rename(columns={'matrix_hansen_sol_parameter': 'matrix_sol_parameter'})
df_raw['original_idx'] = df_raw.index
df_clean = df_raw.dropna().reset_index(drop=True)

LOG_FEATS = ['tortuosity', 'interface_rot', 'Half life_matrix_avg',
             'matrix_CED', 'fiber_dia_or_average_particle_size_mm']
df_log = df_clean.copy()
for f in LOG_FEATS:
    df_log[f] = np.log(df_log[f].clip(lower=1e-10))

PRELIM = [0, 8, 18]
df_58 = df_log[~df_log['original_idx'].isin(PRELIM)].reset_index(drop=True)

RETAINED = ['logP_matrix', 'tortuosity', 'interface_rot', 'matrix_CED',
            'Hemicellulose_Percentage', 'fiber_sol_parameter',
            'fiber_dia_or_average_particle_size_mm', 'Half life_matrix_avg']

X_raw = df_58[RETAINED].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

def kennard_stone(X, n_select):
    n = len(X)
    dists = cdist(X, X)
    i, j = np.unravel_index(np.argmax(dists), dists.shape)
    selected = [int(i), int(j)]
    remaining = [k for k in range(n) if k not in selected]
    while len(selected) < n_select:
        min_d = np.array([np.min(dists[r, selected]) for r in remaining])
        best = remaining[int(np.argmax(min_d))]
        selected.append(best); remaining.remove(best)
    return selected

ext_local = kennard_stone(X_scaled, 9)
train_pool = [i for i in range(len(df_58)) if i not in ext_local]

# Box-Cox Hixson target
y_hix_bc, lam_hix = boxcox(df_58['hixson_rate'].values)

# Identify the 9 worst LOO residuals on the training pool
kernel_init = (C(1.0, (1e-3, 1e3))
               * RBF(length_scale=1.0, length_scale_bounds=(0.5, 20.0))
               + WhiteKernel(noise_level=0.1, noise_level_bounds=(1e-5, 1.0)))
loo_resid = np.zeros(len(train_pool))
for tr, te in LeaveOneOut().split(X_scaled[train_pool]):
    g = GaussianProcessRegressor(kernel=kernel_init, alpha=0.1, normalize_y=True,
                                  n_restarts_optimizer=10, random_state=RANDOM_STATE)
    g.fit(X_scaled[train_pool][tr], y_hix_bc[train_pool][tr])
    loo_resid[te[0]] = y_hix_bc[train_pool][te[0]] - g.predict(X_scaled[train_pool][te])[0]
worst9 = np.argsort(np.abs(loo_resid))[-9:].tolist()
removed_local = [train_pool[k] for k in worst9]
train_final = [k for k in train_pool if k not in removed_local]

print(f"Pipeline reproduced: {len(train_final)} training composites")

X_final  = X_scaled[train_final]
y_final  = y_hix_bc[train_final]
logP_col = RETAINED.index('logP_matrix')

# ============================================================================
# 2. LOO predictions: matrix-identity (logP-quintile) baseline + full GPR
# ============================================================================
def logP_quintile_predict(X, y, col_idx):
    """LOO predictions for the matrix-identity proxy: bin by logP_matrix
    quintile (edges from training fold only) and predict bin mean."""
    preds = np.zeros_like(y, dtype=float)
    for tr, te in LeaveOneOut().split(X):
        logP_tr = X[tr, col_idx]
        edges = np.quantile(logP_tr, [0.0, 0.2, 0.4, 0.6, 0.8, 1.0])
        edges[0] -= 1e-6; edges[-1] += 1e-6
        bin_tr = np.clip(np.searchsorted(edges, logP_tr, side='right') - 1, 0, 4)
        bin_mean = np.array([
            y[tr][bin_tr == b].mean() if (bin_tr == b).any() else y[tr].mean()
            for b in range(5)])
        logP_te = X[te, col_idx]
        bin_te = np.clip(np.searchsorted(edges, logP_te, side='right') - 1, 0, 4)
        preds[te[0]] = bin_mean[bin_te[0]]
    return preds

def gpr_loo_predict(X, y):
    """LOO predictions for the full GPR."""
    preds = np.zeros_like(y, dtype=float)
    kernel = (C(1.0, (1e-3, 1e3))
              * RBF(length_scale=1.0, length_scale_bounds=(0.5, 20.0))
              + WhiteKernel(noise_level=0.1, noise_level_bounds=(1e-5, 1.0)))
    for tr, te in LeaveOneOut().split(X):
        g = GaussianProcessRegressor(kernel=kernel, alpha=0.1, normalize_y=True,
                                      n_restarts_optimizer=10,
                                      random_state=RANDOM_STATE)
        g.fit(X[tr], y[tr])
        preds[te[0]] = g.predict(X[te])[0]
    return preds

quint_pred = logP_quintile_predict(X_final, y_final, logP_col)
gpr_pred   = gpr_loo_predict(X_final, y_final)

abs_err_quint = np.abs(y_final - quint_pred)
abs_err_gpr   = np.abs(y_final - gpr_pred)
rescue        = abs_err_quint - abs_err_gpr   # positive = GPR is better

# Per-composite logP_matrix for the colourbar
logP_per_composite = df_58.iloc[train_final]['logP_matrix'].values

# Identify the two biggest GPR rescues for annotation
top_rescued_idx = np.argsort(rescue)[-2:]

print(f"Mean |residual| under matrix-identity baseline: {abs_err_quint.mean():.3f}")
print(f"Mean |residual| under full GPR:                  {abs_err_gpr.mean():.3f}")
print(f"Fraction of composites rescued by GPR:           {(rescue > 0).mean():.1%}")
print(f"Largest single-composite improvement:            {rescue.max():.3f} "
      f"(logP_m = {logP_per_composite[np.argmax(rescue)]:.2f})")

# ============================================================================
# 3. Render the single-panel figure
# ============================================================================
fig, ax = plt.subplots(figsize=(7.5, 6.5))
fig.patch.set_facecolor('white')

vmin = float(np.nanmin(logP_per_composite))
vmax = float(np.nanmax(logP_per_composite))
sc = ax.scatter(abs_err_quint, abs_err_gpr,
                 c=logP_per_composite, cmap='viridis',
                 vmin=vmin, vmax=vmax,
                 s=85, edgecolor='black', linewidth=0.6, zorder=3)

# Identity diagonal
ax_lim_max = max(abs_err_quint.max(), abs_err_gpr.max()) * 1.10
ax.plot([0, ax_lim_max], [0, ax_lim_max], color='black', lw=1.0,
        ls='--', alpha=0.7, zorder=2,
        label='No improvement\n(identity line)')

# Fibre-green wash over the GPR-improves region (below the diagonal)
poly = mpatches.Polygon([[0, 0], [ax_lim_max, 0], [ax_lim_max, ax_lim_max]],
                        facecolor=FIBRE_GREEN, alpha=0.07, zorder=1)
ax.add_patch(poly)

# "GPR improves over matrix proxy (XX% of composites)" annotation
ax.text(ax_lim_max * 0.78, ax_lim_max * 0.08,
        f'GPR improves over\nmatrix proxy\n'
        f'({(rescue > 0).mean()*100:.0f}% of composites)',
        ha='center', va='center', fontsize=12, color=FIBRE_GREEN,
        fontweight='bold', style='italic')

# Annotate the two biggest individual rescues
for k in top_rescued_idx:
    ax.annotate(f'logP$_m$ = {logP_per_composite[k]:.2f}',
                xy=(abs_err_quint[k], abs_err_gpr[k]),
                xytext=(-65, 30), textcoords='offset points',
                fontsize=12, color='black',
                arrowprops=dict(arrowstyle='-', color='gray', lw=0.6,
                                 connectionstyle='arc3,rad=0.2'))

ax.set_xlim(0, ax_lim_max)
ax.set_ylim(0, ax_lim_max)
ax.set_aspect('equal', adjustable='box')
ax.set_xlabel(r'$|$residual$|$ under matrix-identity baseline', fontsize=18)
ax.set_ylabel(r'$|$residual$|$ under full GPR', fontsize=18)
ax.legend(loc='upper left', frameon=True, edgecolor='black', fontsize=16)

# Clean style: no grid, solid black axes, white background, hide top/right spines
for s in ('top', 'right'):
    ax.spines[s].set_visible(False)
for s in ('left', 'bottom'):
    ax.spines[s].set_color('black')
    ax.spines[s].set_linewidth(1.0)
ax.tick_params(colors='black', labelsize=18)
ax.grid(False)
ax.set_facecolor('white')

# Colourbar
cbar = fig.colorbar(sc, ax=ax, shrink=0.78, pad=0.02)
cbar.set_label(r'log$P_{\rm matrix}$', fontsize=18)
cbar.ax.tick_params(labelsize=9)
cbar.outline.set_edgecolor('black')

plt.tight_layout()
plt.savefig(OUT_PNG, dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig(OUT_PDF, bbox_inches='tight', facecolor='white')
plt.show()

## Calibration from Kuleshov's paper's method

In [ ]:
# ============================================================================
# Calibration metric verification — self-contained drop-in cell
# ============================================================================
# Methodology reference:
#   Kuleshov, V., Fenner, N. & Ermon, S. (2018). Accurate uncertainties for
#   deep learning using calibrated regression. ICML, PMLR 80, 2796-2804.
# ============================================================================

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm, boxcox
from scipy.spatial.distance import cdist
from sklearn.preprocessing import StandardScaler
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C, WhiteKernel
from sklearn.model_selection import LeaveOneOut

# ---- Configuration ----------------------------------------------------------
DATASET_PATH = 'dataset.xlsx'
TARGET_NAME  ='first_order_rate'   # change to 'first_order_rate' for the other target
OUT_PNG      = f'calibration_curve_{TARGET_NAME}.png'
OUT_PDF      = f'calibration_curve_{TARGET_NAME}.pdf'
RANDOM_STATE = 42

MATRIX_BLUE = '#185FA5'
FIBRE_GREEN = '#3B6D11'
ACCENT_RED  = '#E24B4A'

# ============================================================================
# 1. Reproduce the pipeline from dataset.xlsx
# ============================================================================
df_raw = pd.read_excel(DATASET_PATH)
df_raw = df_raw.rename(columns={'matrix_hansen_sol_parameter': 'matrix_sol_parameter'})
df_raw['original_idx'] = df_raw.index

df_clean = df_raw.dropna().reset_index(drop=True)

LOG_FEATS = ['tortuosity', 'interface_rot', 'Half life_matrix_avg',
             'matrix_CED', 'fiber_dia_or_average_particle_size_mm']
df_log = df_clean.copy()
for f in LOG_FEATS:
    df_log[f] = np.log(df_log[f].clip(lower=1e-10))

PRELIM = [0, 8, 18]
df_58 = df_log[~df_log['original_idx'].isin(PRELIM)].reset_index(drop=True)

RETAINED = ['logP_matrix', 'tortuosity', 'interface_rot', 'matrix_CED',
            'Hemicellulose_Percentage', 'fiber_sol_parameter',
            'fiber_dia_or_average_particle_size_mm', 'Half life_matrix_avg']

X_raw = df_58[RETAINED].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

def kennard_stone(X, n_select):
    n = len(X)
    dists = cdist(X, X)
    i, j = np.unravel_index(np.argmax(dists), dists.shape)
    selected = [int(i), int(j)]
    remaining = [k for k in range(n) if k not in selected]
    while len(selected) < n_select:
        min_d = np.array([np.min(dists[r, selected]) for r in remaining])
        best = remaining[int(np.argmax(min_d))]
        selected.append(best); remaining.remove(best)
    return selected

ext_local = kennard_stone(X_scaled, 9)
train_pool = [i for i in range(len(df_58)) if i not in ext_local]

# Box-Cox the target
y_full, lam = boxcox(df_58[TARGET_NAME].values)

# Identify the 9 worst LOO residuals on the training pool
kernel_init = (C(1.0, (1e-3, 1e3))
               * RBF(length_scale=1.0, length_scale_bounds=(0.5, 20.0))
               + WhiteKernel(noise_level=0.1, noise_level_bounds=(1e-5, 1.0)))
loo_resid = np.zeros(len(train_pool))
for tr, te in LeaveOneOut().split(X_scaled[train_pool]):
    g = GaussianProcessRegressor(kernel=kernel_init, alpha=0.1, normalize_y=True,
                                  n_restarts_optimizer=10,
                                  random_state=RANDOM_STATE)
    g.fit(X_scaled[train_pool][tr], y_full[train_pool][tr])
    loo_resid[te[0]] = y_full[train_pool][te[0]] - g.predict(X_scaled[train_pool][te])[0]
worst9 = np.argsort(np.abs(loo_resid))[-9:].tolist()
removed_local = [train_pool[k] for k in worst9]
train_final = [k for k in train_pool if k not in removed_local]

X_train = X_scaled[train_final]
y_train = y_full[train_final]
n_train, n_feat = X_train.shape

print(f"[setup] Pipeline reproduced: {n_train} training × {n_feat} features ({TARGET_NAME})")

# ============================================================================
# 2. Fit final GPR and get LOO predictions with predictive standard deviations
# ============================================================================
gpr_full = GaussianProcessRegressor(kernel=kernel_init, alpha=0.1, normalize_y=True,
                                      n_restarts_optimizer=10,
                                      random_state=RANDOM_STATE)
gpr_full.fit(X_train, y_train)
print(f"[setup] Fitted GPR with kernel: {gpr_full.kernel_}")

# LOO with the fitted hyperparameters (no per-fold re-optimisation)
print("[loo] Computing leave-one-out predictions and predictive variances...")
mu_loo = np.zeros(n_train)
sigma_loo = np.zeros(n_train)
fitted_kernel = gpr_full.kernel_

for tr_idx, te_idx in LeaveOneOut().split(X_train):
    g = GaussianProcessRegressor(
        kernel=fitted_kernel, alpha=0.1, normalize_y=True,
        optimizer=None, random_state=RANDOM_STATE)
    g.fit(X_train[tr_idx], y_train[tr_idx])
    mu, sd = g.predict(X_train[te_idx], return_std=True)
    mu_loo[te_idx[0]] = mu[0]
    sigma_loo[te_idx[0]] = sd[0]

q2_loo = 1.0 - np.sum((y_train - mu_loo)**2) / np.sum((y_train - y_train.mean())**2)
print(f"[loo] LOO Q² = {q2_loo:.3f}")

# ============================================================================
# 3. Empirical-vs-nominal coverage at 19 confidence levels (5% increments)
# ============================================================================
nominal_levels = np.linspace(0.05, 0.95, 19)
empirical = np.zeros_like(nominal_levels)

for j, p in enumerate(nominal_levels):
    z = norm.ppf((1 + p) / 2)   # two-sided interval half-width in σ-units
    lower = mu_loo - z * sigma_loo
    upper = mu_loo + z * sigma_loo
    inside = (y_train >= lower) & (y_train <= upper)
    empirical[j] = inside.mean()

# Also compute the Kuleshov 10-bin form (directly comparable to their Table 1)
ten_nominal = np.linspace(0.1, 0.99, 10)
ten_empirical = np.zeros_like(ten_nominal)
for j, p in enumerate(ten_nominal):
    z = norm.ppf((1 + p) / 2)
    lower = mu_loo - z * sigma_loo
    upper = mu_loo + z * sigma_loo
    ten_empirical[j] = ((y_train >= lower) & (y_train <= upper)).mean()

# ============================================================================
# 4. Both calibration error metrics
# ============================================================================
L1_cal_error      = np.mean(np.abs(empirical - nominal_levels))
sq_cal_error_sum  = np.sum((empirical - nominal_levels) ** 2)
sq_cal_error_mean = np.mean((empirical - nominal_levels) ** 2)
rmse_cal_error    = np.sqrt(sq_cal_error_mean)
sq_cal_10bin      = np.sum((ten_empirical - ten_nominal) ** 2)

print("\n" + "=" * 70)
print("CALIBRATION ERROR METRICS")
print("=" * 70)
print(f"  L1 (mean absolute deviation):                {L1_cal_error:.4f}")
print(f"  Squared sum form, 19 bins:                   {sq_cal_error_sum:.4f}")
print(f"  Squared sum form, 10 bins (Kuleshov-style):  {sq_cal_10bin:.4f}")
print(f"  Squared mean form, 19 bins:                  {sq_cal_error_mean:.4f}")
print(f"  RMSE form (sqrt of mean squared):            {rmse_cal_error:.4f}")
print()
print(f"  Empirical coverage at 95% nominal:           {empirical[-1]:.3f}")
print(f"  Empirical coverage at 50% nominal:           {empirical[len(empirical)//2]:.3f}")

# ============================================================================
# 5. Compare to Kuleshov 2018 Table 1
# ============================================================================
print("\n" + "=" * 70)
print("COMPARISON TO KULESHOV ET AL. 2018 (TABLE 1)")
print("=" * 70)
print("Their reported calibration error ranges across 8 UCI datasets:")
print("  Recalibrated Bayesian Linear Regression:    0.006 – 0.083")
print("  Recalibrated Approx. Bayesian NN:            0.015 – 0.065")
print("  Uncalibrated Bayesian Linear Regression:    0.024 – 0.175")
print("  Uncalibrated Approx. Bayesian NN:            0.043 – 0.205")
print("  Concrete Dropout:                            0.039 – 0.228")
print("  Deep Ensemble:                               0.027 – 0.242")
print()
print(f"Our squared cal error (10-bin, directly comparable): {sq_cal_10bin:.4f}")

if sq_cal_10bin < 0.05:
    interpretation = "well within the recalibrated-model range"
elif sq_cal_10bin < 0.083:
    interpretation = "comparable to recalibrated models in their experiments"
elif sq_cal_10bin < 0.175:
    interpretation = "comparable to typical uncalibrated Bayesian models"
else:
    interpretation = "higher than typical — recalibration would benefit the model"
print(f"Interpretation: {interpretation}")

# ============================================================================
# 6. Plot the calibration curve
# ============================================================================
fig, ax = plt.subplots(figsize=(7, 6.5))
fig.patch.set_facecolor('white')

ax.plot([0, 1], [0, 1], color='black', lw=1.2, ls='--', alpha=0.7,
         label='Perfect calibration', zorder=1)
ax.plot(nominal_levels, empirical, color=MATRIX_BLUE, lw=2.0,
         marker='o', markersize=7, markerfacecolor=MATRIX_BLUE,
         markeredgecolor='black', markeredgewidth=0.5,
         label=f'GPR ({TARGET_NAME})', zorder=3)
ax.fill_between(nominal_levels, nominal_levels, empirical,
                 alpha=0.15, color=MATRIX_BLUE, zorder=2)
ax.scatter([0.95], [empirical[-1]], s=140, marker='o',
            facecolor='white', edgecolor=ACCENT_RED, linewidth=2.0,
            zorder=4, label=f'95% nominal: {empirical[-1]*100:.1f}% empirical')

metrics_text = (
    f'L1 (mean abs dev):           {L1_cal_error:.3f}\n'
    f'Squared (10-bin, Kuleshov):  {sq_cal_10bin:.3f}\n'
    f'Squared (19-bin):            {sq_cal_error_sum:.3f}\n'
    f'RMSE:                        {rmse_cal_error:.3f}'
)
ax.text(0.04, 0.9, metrics_text, transform=ax.transAxes,
         fontsize=12, family='serif', verticalalignment='top',
         bbox=dict(boxstyle='round,pad=0.4', facecolor='#F5F5F5',
                   edgecolor='black', linewidth=0.5))

ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_aspect('equal', adjustable='box')
ax.set_xlabel('Nominal (expected) confidence level', fontsize=18)
ax.set_ylabel('Empirical (observed) confidence level', fontsize=18)
#ax.set_title(f'Calibration curve — {TARGET_NAME}\n'
              #f'Methodology: Kuleshov, Fenner & Ermon (2018), ICML',
              #fontsize=11, pad=10)
ax.legend(loc='lower right', frameon=True, edgecolor='black', fontsize=12)

for s in ('top', 'right'):  ax.spines[s].set_visible(False)
for s in ('left', 'bottom'): ax.spines[s].set_color('black'); ax.spines[s].set_linewidth(1.0)
ax.tick_params(colors='black', labelsize=10)
ax.grid(False); ax.set_facecolor('white')

plt.tight_layout()
plt.savefig(OUT_PNG, dpi=6000, bbox_inches='tight', facecolor='white')
#plt.savefig(OUT_PDF,  dpi=20000, bbox_inches='tight', facecolor='white')
plt.show()
print(f"\n[output] Saved {OUT_PNG} and {OUT_PDF}")

# ============================================================================


In [ ]:
# =============================================================================
# Calibration numerical details  ->  Excel workbook
# =============================================================================
# Drop-in. Recomputes the uncertainty calibration of the GPR for BOTH targets
# from dataset.xlsx (via reproduce_pipeline) and writes every number to a
# formatted .xlsx workbook.
#
# Workbook sheets:
#   - Summary          : headline metrics, both targets side by side
#   - FirstOrder_bins  : per-bin nominal vs empirical coverage (first-order)
#   - Hixson_bins      : per-bin nominal vs empirical coverage (Hixson)
#
# Metrics computed (per target):
#   - Observed coverage at 50/68/90/95% nominal prediction intervals
#   - L1 calibration error (mean |nominal - empirical|)  -- manuscript number
#   - Kuleshov squared calibration error, 10-bin and 19-bin (Eq. 9, w_j = 1)
#   - Sharpness: mean predictive sigma, and mean predictive variance
#   - RMSE of LOO predictions
#
# The LOO predictive mean and std are obtained by refitting the GPR on each
# leave-one-out fold (kernel hyper-parameters fixed at the full-data fit, as in
# the manuscript's calibration analysis).
#
# Cost: ~2-4 minutes on real data (LOO refits for both targets).
# =============================================================================

import warnings
warnings.filterwarnings('ignore')

import copy
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneOut
from sklearn.gaussian_process import GaussianProcessRegressor
from scipy import stats

from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

from colored_analyses_b2_b6 import (
    reproduce_pipeline, make_add_interactions, default_gpr_factory,
)

# ---- Configuration ----------------------------------------------------------
DATASET_PATH = 'dataset.xlsx'
OUT_XLSX     = 'calibration_numerical_details.xlsx'
NOMINAL_LEVELS = [0.50, 0.68, 0.90, 0.95]   # headline coverage checkpoints

# ============================================================================
# 1. LOO predictive mean + std for one target
# ============================================================================
def loo_predictions(X_train_base, y_train_bc, feat_names):
    """Leave-one-out predictive mean and std in Box-Cox space.
    Kernel hyper-parameters are fixed at the full-data fit (optimizer=None on
    the folds), matching the manuscript's calibration protocol."""
    add_int = make_add_interactions(feat_names)
    X_int = add_int(X_train_base)
    sc = StandardScaler().fit(X_int)
    Xs = sc.transform(X_int)

    # Full-data fit to lock in kernel hyper-parameters
    gpr_full = copy.deepcopy(default_gpr_factory())
    gpr_full.fit(Xs, y_train_bc)
    fitted_kernel = gpr_full.kernel_

    n = len(y_train_bc)
    mu = np.zeros(n)
    sigma = np.zeros(n)
    for tr, te in LeaveOneOut().split(Xs):
        g = GaussianProcessRegressor(kernel=fitted_kernel, alpha=gpr_full.alpha,
                                      normalize_y=True, optimizer=None)
        g.fit(Xs[tr], y_train_bc[tr])
        m, s = g.predict(Xs[te], return_std=True)
        mu[te[0]] = m[0]
        sigma[te[0]] = s[0]
    return mu, sigma

# ============================================================================
# 2. Calibration metrics from LOO mean/std and observed values
# ============================================================================
def calibration_metrics(y_true, mu, sigma, n_bins):
    """Return per-bin nominal vs empirical coverage and the scalar metrics.
    Coverage is computed from the central prediction interval implied by a
    Gaussian predictive distribution N(mu, sigma^2)."""
    # z-score of each observation under its predictive distribution
    z = (y_true - mu) / sigma
    # Empirical CDF value of each point: Phi(z)
    p_obs = stats.norm.cdf(z)

    # For a set of nominal quantile levels q, the expected fraction of points
    # with p_obs <= q is q itself (Kuleshov 2018, Eq. 8-9).
    q_levels = np.linspace(1.0 / n_bins, 1.0, n_bins)
    emp = np.array([(p_obs <= q).mean() for q in q_levels])

    # L1 calibration error: mean |q - emp|
    l1 = np.mean(np.abs(q_levels - emp))
    # Kuleshov squared calibration error (Eq. 9, w_j = 1): sum (q - emp)^2
    cal_sq = np.sum((q_levels - emp) ** 2)
    return q_levels, emp, l1, cal_sq

def central_interval_coverage(y_true, mu, sigma, level):
    """Fraction of observations inside the central `level` prediction interval."""
    half = stats.norm.ppf(0.5 + level / 2.0)   # e.g. level 0.95 -> 1.96
    lo = mu - half * sigma
    hi = mu + half * sigma
    return np.mean((y_true >= lo) & (y_true <= hi))

# ============================================================================
# 3. Run both targets
# ============================================================================
results = {}
for tgt_key, tgt_label in [('first_order_rate', 'First-order'),
                            ('hixson_rate',      'Hixson-Crowell')]:
    print(f"[calibration] {tgt_label}: reproducing pipeline ...")
    pipe = reproduce_pipeline(DATASET_PATH, target=tgt_key, verbose=False)
    Xb   = pipe['X_train_base']
    y_bc = pipe['y_train_bc']
    y_log = pipe['y_train_log']
    feats = pipe['feat_names']

    print(f"[calibration] {tgt_label}: computing LOO predictive distribution ...")
    mu, sigma = loo_predictions(Xb, y_bc, feats)

    # Observed values in the same (Box-Cox) space the GPR was fit in
    y_true = y_bc

    # Per-bin coverage at 10 and 19 bins (Kuleshov uses both forms)
    q10, emp10, l1_10, sq10 = calibration_metrics(y_true, mu, sigma, n_bins=10)
    q19, emp19, l1_19, sq19 = calibration_metrics(y_true, mu, sigma, n_bins=19)

    # Headline coverage at the nominal levels
    coverage = {lvl: central_interval_coverage(y_true, mu, sigma, lvl)
                for lvl in NOMINAL_LEVELS}

    # Sharpness + RMSE
    sharpness_sigma = float(np.mean(sigma))
    sharpness_var   = float(np.mean(sigma ** 2))
    rmse = float(np.sqrt(np.mean((y_true - mu) ** 2)))

    results[tgt_label] = dict(
        n=len(y_true),
        coverage=coverage,
        l1_10=l1_10, sq10=sq10,
        l1_19=l1_19, sq19=sq19,
        sharpness_sigma=sharpness_sigma,
        sharpness_var=sharpness_var,
        rmse=rmse,
        q10=q10, emp10=emp10,
        q19=q19, emp19=emp19,
    )
    print(f"[calibration] {tgt_label}: done "
          f"(L1 10-bin = {l1_10:.3f}, squared 10-bin = {sq10:.3f})")

# ============================================================================
# 4. Write the Excel workbook
# ============================================================================
wb = Workbook()

# --- styling helpers ---------------------------------------------------------
FONT_NAME   = 'Arial'
hdr_fill    = PatternFill('solid', start_color='185FA5')
hdr_font    = Font(name=FONT_NAME, bold=True, color='FFFFFF', size=11)
title_font  = Font(name=FONT_NAME, bold=True, size=13, color='000000')
label_font  = Font(name=FONT_NAME, bold=True, size=11, color='000000')
val_font    = Font(name=FONT_NAME, size=11, color='000000')
note_font   = Font(name=FONT_NAME, italic=True, size=9, color='555555')
thin        = Side(style='thin', color='BBBBBB')
border      = Border(left=thin, right=thin, top=thin, bottom=thin)
centre      = Alignment(horizontal='center', vertical='center')
left_al     = Alignment(horizontal='left',   vertical='center')

def style_header_row(ws, row, ncols):
    for c in range(1, ncols + 1):
        cell = ws.cell(row=row, column=c)
        cell.fill = hdr_fill; cell.font = hdr_font
        cell.alignment = centre; cell.border = border

# --- Sheet 1: Summary --------------------------------------------------------
ws = wb.active
ws.title = 'Summary'
labels = list(results.keys())   # ['First-order', 'Hixson-Crowell']

ws['A1'] = 'GPR uncertainty calibration — numerical summary'
ws['A1'].font = title_font
ws.merge_cells('A1:C1')

row = 3
ws.cell(row=row, column=1, value='Metric')
ws.cell(row=row, column=2, value=labels[0])
ws.cell(row=row, column=3, value=labels[1])
style_header_row(ws, row, 3)

summary_rows = []
summary_rows.append(('Training samples (n)',
                     results[labels[0]]['n'], results[labels[1]]['n'], '0'))
for lvl in NOMINAL_LEVELS:
    summary_rows.append((f'Observed coverage at {int(lvl*100)}% nominal',
                         results[labels[0]]['coverage'][lvl],
                         results[labels[1]]['coverage'][lvl], '0.0%'))
summary_rows.append(('L1 calibration error (10-bin)',
                     results[labels[0]]['l1_10'], results[labels[1]]['l1_10'], '0.000'))
summary_rows.append(('Squared calibration error (10-bin, Kuleshov Eq. 9)',
                     results[labels[0]]['sq10'], results[labels[1]]['sq10'], '0.000'))
summary_rows.append(('L1 calibration error (19-bin)',
                     results[labels[0]]['l1_19'], results[labels[1]]['l1_19'], '0.000'))
summary_rows.append(('Squared calibration error (19-bin, Kuleshov Eq. 9)',
                     results[labels[0]]['sq19'], results[labels[1]]['sq19'], '0.000'))
summary_rows.append(('Sharpness — mean predictive sigma',
                     results[labels[0]]['sharpness_sigma'],
                     results[labels[1]]['sharpness_sigma'], '0.000'))
summary_rows.append(('Sharpness — mean predictive variance',
                     results[labels[0]]['sharpness_var'],
                     results[labels[1]]['sharpness_var'], '0.0000'))
summary_rows.append(('RMSE of LOO predictions (Box-Cox space)',
                     results[labels[0]]['rmse'], results[labels[1]]['rmse'], '0.000'))

for i, (label, v0, v1, numfmt) in enumerate(summary_rows):
    r = row + 1 + i
    c1 = ws.cell(row=r, column=1, value=label)
    c1.font = label_font; c1.alignment = left_al; c1.border = border
    for col, v in [(2, v0), (3, v1)]:
        cell = ws.cell(row=r, column=col, value=v)
        cell.font = val_font; cell.alignment = centre; cell.border = border
        cell.number_format = numfmt

note_r = row + 1 + len(summary_rows) + 1
ws.cell(row=note_r, column=1,
        value=('Kuleshov 2018 Table 1: recalibrated-model squared calibration '
               'error 0.006-0.083; uncalibrated 0.024-0.242. '
               'Coverage computed from the Gaussian LOO predictive distribution.'))
ws.cell(row=note_r, column=1).font = note_font
ws.merge_cells(start_row=note_r, start_column=1, end_row=note_r, end_column=3)

ws.column_dimensions['A'].width = 48
ws.column_dimensions['B'].width = 18
ws.column_dimensions['C'].width = 18

# --- Per-target bin sheets ---------------------------------------------------
def write_bin_sheet(sheet_name, res):
    ws = wb.create_sheet(sheet_name)
    ws['A1'] = f'{sheet_name} — per-bin calibration (nominal vs empirical)'
    ws['A1'].font = title_font
    ws.merge_cells('A1:D1')

    # 10-bin block
    ws['A3'] = '10-bin calibration curve'
    ws['A3'].font = label_font
    hdr = ['Bin', 'Nominal coverage', 'Empirical coverage', 'Deviation (emp - nom)']
    for c, h in enumerate(hdr, start=1):
        ws.cell(row=4, column=c, value=h)
    style_header_row(ws, 4, 4)
    for i, (q, e) in enumerate(zip(res['q10'], res['emp10'])):
        r = 5 + i
        for c, (v, fmt) in enumerate(
                [(i + 1, '0'), (q, '0.000'), (e, '0.000'), (e - q, '0.000')],
                start=1):
            cell = ws.cell(row=r, column=c, value=v)
            cell.font = val_font; cell.alignment = centre; cell.border = border
            cell.number_format = fmt

    # 19-bin block, offset to the right
    ws['F3'] = '19-bin calibration curve'
    ws['F3'].font = label_font
    for c, h in enumerate(hdr, start=6):
        ws.cell(row=4, column=c, value=h)
    style_header_row(ws, 4, 9)
    for i, (q, e) in enumerate(zip(res['q19'], res['emp19'])):
        r = 5 + i
        for c, (v, fmt) in enumerate(
                [(i + 1, '0'), (q, '0.000'), (e, '0.000'), (e - q, '0.000')],
                start=6):
            cell = ws.cell(row=r, column=c, value=v)
            cell.font = val_font; cell.alignment = centre; cell.border = border
            cell.number_format = fmt

    for col in ['A', 'B', 'C', 'D', 'F', 'G', 'H', 'I']:
        ws.column_dimensions[col].width = 20

write_bin_sheet('FirstOrder_bins', results['First-order'])
write_bin_sheet('Hixson_bins',     results['Hixson-Crowell'])

wb.save(OUT_XLSX)
print(f"\n[output] Saved {OUT_XLSX}")

# ============================================================================
# 5. Also echo the headline numbers to the notebook output
# ============================================================================
print("\n" + "=" * 70)
print("CALIBRATION SUMMARY")
print("=" * 70)
for label in labels:
    r = results[label]
    print(f"\n{label}  (n = {r['n']}):")
    for lvl in NOMINAL_LEVELS:
        print(f"  coverage at {int(lvl*100):>2d}% nominal : {r['coverage'][lvl]*100:5.1f}%")
    print(f"  L1 calibration error (10-bin)      : {r['l1_10']:.3f}")
    print(f"  squared cal. error (10-bin)        : {r['sq10']:.3f}")
    print(f"  L1 calibration error (19-bin)      : {r['l1_19']:.3f}")
    print(f"  squared cal. error (19-bin)        : {r['sq19']:.3f}")
    print(f"  sharpness (mean sigma)             : {r['sharpness_sigma']:.3f}")
    print(f"  RMSE LOO (Box-Cox space)           : {r['rmse']:.3f}")

## Biplot (Fig 1)

In [ ]:
# ============================================================================
# Figure 1 — Information-dense PCA biplot
# ============================================================================
# pipeline end-to-end so the role assignments (training / external /
# removed-outlier) are identical to those used in the modelling notebook.
#
# Output: a single PCA biplot saved as PNG (300 dpi) +
# PDF (vector). Run this cell after the existing PCA section; it does not
# depend on any variables defined in earlier cells.
# ============================================================================

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C, WhiteKernel
from sklearn.model_selection import LeaveOneOut
from scipy.stats import boxcox
from scipy.spatial.distance import cdist

# ---- Edit only this section if your dataset path or output names differ ----
DATASET_PATH = 'dataset.xlsx'
OUT_PNG      = 'Figure_1_PCA_biplot.png'
OUT_PDF      = 'Figure_1_PCA_biplot.pdf'
RANDOM_STATE = 42
# ----------------------------------------------------------------------------

# Step 1: load + drop NaN
df_raw = pd.read_excel(DATASET_PATH)
df_raw = df_raw.rename(columns={'matrix_hansen_sol_parameter': 'matrix_sol_parameter'})
df_raw['original_idx'] = df_raw.index
df_clean = df_raw.dropna().reset_index(drop=True)

# Step 2: log-transform 5 right-skewed features
LOG_FEATS = ['tortuosity', 'interface_rot', 'Half life_matrix_avg',
             'matrix_CED', 'fiber_dia_or_average_particle_size_mm']
df_log = df_clean.copy()
for f in LOG_FEATS:
    df_log[f] = np.log(df_log[f].clip(lower=1e-10))

# Step 3: drop 3 preliminary outliers (original indices 0, 8, 18)
PRELIM = [0, 8, 18]
df_58 = df_log[~df_log['original_idx'].isin(PRELIM)].reset_index(drop=True)

# Step 4: 8 retained features after collinearity filtering
RETAINED = ['logP_matrix', 'tortuosity', 'interface_rot', 'matrix_CED',
            'Hemicellulose_Percentage', 'fiber_sol_parameter',
            'fiber_dia_or_average_particle_size_mm', 'Half life_matrix_avg']

PRETTY = {
    'logP_matrix': r'log$P_{\rm matrix}$',
    'tortuosity': r'log(tortuosity)',
    'interface_rot': r'log(interface_rot)',
    'matrix_CED': r'log(matrix CED)',
    'Hemicellulose_Percentage': 'Hemicellulose %',
    'fiber_sol_parameter': r'$\delta_{\rm fibre}$',
    'fiber_dia_or_average_particle_size_mm': r'log(fibre diam.)',
    'Half life_matrix_avg': r'log(matrix $t_{1/2}$)',
}

X_raw = df_58[RETAINED].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

# Step 5: Kennard-Stone external split (9 most diverse composites)
def kennard_stone(X, n_select):
    n = len(X)
    dists = cdist(X, X)
    i, j = np.unravel_index(np.argmax(dists), dists.shape)
    selected = [int(i), int(j)]
    remaining = [k for k in range(n) if k not in selected]
    while len(selected) < n_select:
        min_d = np.array([np.min(dists[r, selected]) for r in remaining])
        best = remaining[int(np.argmax(min_d))]
        selected.append(best); remaining.remove(best)
    return selected

ext_local = kennard_stone(X_scaled, 9)
train_pool_local = [i for i in range(len(df_58)) if i not in ext_local]

# Step 6: Box-Cox the Hixson target on the 58-row frame
y_hix_bc, lam_hix = boxcox(df_58['hixson_rate'].values)

# Step 7: LOO residuals on the 49-sample training pool; identify 9 worst
y_pool = y_hix_bc[train_pool_local]
X_pool = X_scaled[train_pool_local]
kernel = (C(1.0, (1e-3, 1e3))
          * RBF(length_scale=1.0, length_scale_bounds=(0.5, 20.0))
          + WhiteKernel(noise_level=0.1, noise_level_bounds=(1e-5, 1.0)))
loo_resid = np.zeros(len(train_pool_local))
for tr, te in LeaveOneOut().split(X_pool):
    gpr = GaussianProcessRegressor(kernel=kernel, alpha=0.1, normalize_y=True,
                                    n_restarts_optimizer=10, random_state=RANDOM_STATE)
    gpr.fit(X_pool[tr], y_pool[tr])
    loo_resid[te[0]] = y_pool[te[0]] - gpr.predict(X_pool[te])[0]
worst9 = np.argsort(np.abs(loo_resid))[-9:].tolist()
removed_local = [train_pool_local[k] for k in worst9]
train_final_local = [k for k in train_pool_local if k not in removed_local]

# Step 8: build role lookup keyed by original row index
role = {}
for k in PRELIM:                role[k] = 'preliminary'
for k in train_final_local:     role[int(df_58.iloc[k]['original_idx'])] = 'training'
for k in ext_local:             role[int(df_58.iloc[k]['original_idx'])] = 'external'
for k in removed_local:         role[int(df_58.iloc[k]['original_idx'])] = 'removed'

# Step 9: PCA on the 58-row scaled matrix
pca = PCA(n_components=2)
X_pc = pca.fit_transform(X_scaled)
ev1, ev2 = pca.explained_variance_ratio_ * 100
loadings = pca.components_.T  # (8, 2)
load_mag = np.sqrt(loadings[:, 0]**2 + loadings[:, 1]**2)
top5_idx = np.argsort(load_mag)[-5:][::-1]
top5_feats = [RETAINED[i] for i in top5_idx]

# Per-point dataframe for plotting
plotdf = df_58.copy()
plotdf['PC1'] = X_pc[:, 0]
plotdf['PC2'] = X_pc[:, 1]
plotdf['log_kHix'] = np.log10(plotdf['hixson_rate'])
plotdf['role'] = plotdf['original_idx'].map(role)

# ---------------------------------------------------------------------------
# Build the figure
# ---------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(9, 7.5))
vmin = plotdf['log_kHix'].min()
vmax = plotdf['log_kHix'].max()

# Marker scheme by pipeline role
ROLES = {
    'training': dict(marker='o', edge='black', lw=0.7, s=110,
                     label=f'Training (n = {sum(plotdf["role"]=="training")})'),
    'external': dict(marker='s', edge='black', lw=1.2, s=130,
                     label=f'External / Kennard\u2013Stone (n = {sum(plotdf["role"]=="external")})'),
    'removed':  dict(marker='X', edge='#444444', lw=0.7, s=130,
                     label=f'Removed outliers (n = {sum(plotdf["role"]=="removed")})'),
}

sc = None
for r, style in ROLES.items():
    sub = plotdf[plotdf['role'] == r]
    if r == 'removed':
        ax.scatter(sub['PC1'], sub['PC2'], marker=style['marker'], s=style['s'],
                   facecolors='#888888', edgecolors=style['edge'],
                   linewidth=style['lw'], alpha=0.85, zorder=3)
    else:
        sc = ax.scatter(sub['PC1'], sub['PC2'], c=sub['log_kHix'], cmap='viridis',
                         vmin=vmin, vmax=vmax, marker=style['marker'], s=style['s'],
                         edgecolors=style['edge'], linewidth=style['lw'],
                         zorder=4 if r == 'external' else 3)

# Loading arrows for top-5 features with displaced labels (anti-overlap)
max_extent = max(np.abs(X_pc).max(axis=0))
arrow_scale = 1.05 * max_extent
arrow_props = dict(arrowstyle='-|>', mutation_scale=15, lw=1.4,
                    color='#B71C1C', alpha=0.92, shrinkA=0, shrinkB=2)
endpoints = np.array([loadings[fidx] * arrow_scale for fidx in top5_idx])

def angular_spread(endpoints, min_angle_deg=20, radial_factor=1.30):
    n = len(endpoints)
    angles = np.arctan2(endpoints[:, 1], endpoints[:, 0])
    radii = np.linalg.norm(endpoints, axis=1)
    order = np.argsort(angles)
    sorted_ang = angles[order].copy()
    min_rad = np.deg2rad(min_angle_deg)
    for _ in range(50):
        changed = False
        for i in range(n):
            j = (i + 1) % n
            gap = (sorted_ang[j] - sorted_ang[i]) % (2 * np.pi)
            if gap < min_rad:
                push = (min_rad - gap) / 2
                sorted_ang[i] -= push; sorted_ang[j] += push
                changed = True
        if not changed: break
    new_angles = np.zeros(n)
    new_angles[order] = sorted_ang
    label_r = radii.max() * radial_factor
    return label_r * np.cos(new_angles), label_r * np.sin(new_angles)

label_x, label_y = angular_spread(endpoints)

for n_arrow, fidx in enumerate(top5_idx):
    dx, dy = endpoints[n_arrow]
    ax.annotate('', xy=(dx, dy), xytext=(0, 0), arrowprops=arrow_props, zorder=5)
    tx, ty = label_x[n_arrow], label_y[n_arrow]
    ax.plot([dx, tx], [dy, ty], color='#B71C1C', lw=0.6, alpha=0.5, zorder=4)
    ax.text(tx, ty, PRETTY[RETAINED[fidx]], fontsize=12, fontweight='bold',
             color='#B71C1C', ha='center', va='center',
             bbox=dict(boxstyle='round,pad=0.28', facecolor='white',
                       edgecolor='#B71C1C', linewidth=0.6, alpha=0.95),
             zorder=6)

ax.axhline(0, color='gray', lw=0.5, alpha=0.5, zorder=1)
ax.axvline(0, color='gray', lw=0.5, alpha=0.5, zorder=1)
ax.set_xlabel(f'PC1 ({ev1:.1f}% of variance)', fontsize=22)
ax.set_ylabel(f'PC2 ({ev2:.1f}% of variance)', fontsize=22)
#ax.tick_params(labelsize=10)

xpad = 0.45 * (plotdf['PC1'].max() - plotdf['PC1'].min())
ypad = 0.45 * (plotdf['PC2'].max() - plotdf['PC2'].min())
ax.set_xlim(plotdf['PC1'].min() - xpad, plotdf['PC1'].max() + xpad)
ax.set_ylim(plotdf['PC2'].min() - ypad, plotdf['PC2'].max() + ypad)

cbar = fig.colorbar(sc, ax=ax, shrink=0.78, pad=0.02)
cbar.set_label(r'$\log_{10}(k_{\rm Hix})$  (day$^{-1}$)', fontsize=22)
#cbar.ax.tick_params(labelsize=9)

handles = [
    Line2D([0],[0], marker='o', color='w', markerfacecolor='#888888',
           markeredgecolor='black', markersize=11, markeredgewidth=0.7,
           label=ROLES['training']['label']),
    Line2D([0],[0], marker='s', color='w', markerfacecolor='#888888',
           markeredgecolor='black', markersize=11, markeredgewidth=1.2,
           label=ROLES['external']['label']),
    Line2D([0],[0], marker='X', color='w', markerfacecolor='#888888',
           markeredgecolor='#444444', markersize=12, markeredgewidth=0.7,
           label=ROLES['removed']['label']),
]
leg = ax.legend(handles=handles, loc='upper left', frameon=True, framealpha=0.92,
                 fontsize=12, borderpad=0.6, handletextpad=0.6,
                 title='Pipeline role', title_fontsize=12)
leg.get_frame().set_edgecolor('#888888')
leg.get_frame().set_linewidth(0.6)

plt.tight_layout()
plt.savefig(OUT_PNG, dpi=300, bbox_inches='tight')
plt.savefig(OUT_PDF, bbox_inches='tight')
plt.show()

# Print the values you need for the figure caption
print(f"PC1 variance captured: {ev1:.1f}%")
print(f"PC2 variance captured: {ev2:.1f}%")
print(f"PC1 + PC2 cumulative:  {ev1 + ev2:.1f}%")
print(f"Top-5 loading features (in order of |loading|):")
for k, fidx in enumerate(top5_idx, 1):
    print(f"   {k}. {RETAINED[fidx]:42s}  |L| = {load_mag[fidx]:.3f}")
print(f"Role counts:  training={sum(plotdf['role']=='training')}, "
      f"external={sum(plotdf['role']=='external')}, "
      f"removed={sum(plotdf['role']=='removed')}")


## Six Panel Tornado Plot for SHAP-bootstrap ranking

In [ ]:
# =============================================================================
# Six-panel tornado composite — SHAP global, top-3 stability, rank-1 stability
# =============================================================================
# Drop-in. Imports run_b5 from colored_analyses_b2_b6.py so the bootstrap
# numbers match the manuscript exactly.
#
# Layout (3 rows x 3 columns; centre column = label strip):
#
#                FO (mirrored)         Feature names         Hixson (normal)
#   Row 0:    [a Mean|SHAP| FO  ]     [  logP_matrix  ]    [b Mean|SHAP| Hix]
#   Row 1:    [c Top-3 freq  FO ]     [  tortuosity   ]    [d Top-3 freq Hix]
#   Row 2:    [e Rank-1 freq FO ]     [  interface_rot]    [f Rank-1 freq Hix]
#
# Single canonical feature ordering applied to ALL rows: Hixson's top-3 set
# frequency descending. So the centre label column is identical in every row
# and a reader can trace any feature's profile straight down the figure.
# All text in black

# =============================================================================

import warnings
warnings.filterwarnings('ignore')

import copy
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Patch
from matplotlib.ticker import FuncFormatter
from sklearn.preprocessing import StandardScaler
import shap

from colored_analyses_b2_b6 import (
    reproduce_pipeline, run_b5,
    make_add_interactions, default_gpr_factory,
    MATRIX_BLUE, FIBRE_GREEN, FEATURE_CLASS,
)

# ---- Configuration ----------------------------------------------------------
DATASET_PATH = 'dataset.xlsx'
OUT_PNG      = 'figure_six_panel_tornado.png'
OUT_PDF      = 'figure_six_panel_tornado.pdf'
OUT_DIR      = './analyses_output'
N_BOOT       = 60
SHAP_NSAMPLES_GLOBAL = 150

# Typography — all text BLACK; sizes large enough for composite scale
TEXT_COLOR   = 'black'
TICK_FS      = 13
LABEL_FS     = 15
VALUE_FS     = 12
LEGEND_FS    = 13
ROWLBL_FS    = 16
FEATNAME_FS  = 14

def feat_color(name: str) -> str:
    cls = FEATURE_CLASS.get(name, 'fibre')
    return MATRIX_BLUE if cls == 'matrix' else FIBRE_GREEN

# ============================================================================
# 1. Canonical pipeline for both targets
# ============================================================================
print("[setup] Reproducing canonical pipeline for first-order target ...")
pipe_fo = reproduce_pipeline(DATASET_PATH, target='first_order_rate', verbose=True)
print("\n[setup] Reproducing canonical pipeline for Hixson target ...")
pipe_hx = reproduce_pipeline(DATASET_PATH, target='hixson_rate', verbose=True)

X_tr_fo, y_tr_fo = pipe_fo['X_train_base'], pipe_fo['y_train_bc']
X_tr_hx, y_tr_hx = pipe_hx['X_train_base'], pipe_hx['y_train_bc']
feat_names = pipe_fo['feat_names']

# ============================================================================
# 2. Headline (non-bootstrap) SHAP for both targets
# ----------------------------------------------------------------------------
# Uses the same machinery run_b5 uses internally (add_interactions +
# fold-internal scaler), so the SHAP row is on the same footing as the
# bootstrap rows.
# ============================================================================
def headline_shap(X_train_base, y_train_bc, label):
    add_int = make_add_interactions(feat_names)
    sc = StandardScaler().fit(add_int(X_train_base))
    gpr = copy.deepcopy(default_gpr_factory())
    gpr.fit(sc.transform(add_int(X_train_base)), y_train_bc)

    def f(X_query_base):
        return gpr.predict(sc.transform(add_int(X_query_base)))

    print(f"[shap-global] {label}: computing baseline SHAP "
          f"(nsamples={SHAP_NSAMPLES_GLOBAL}) ...")
    explainer = shap.KernelExplainer(
        f, X_train_base[:min(20, len(X_train_base))], silent=True)
    sv = explainer.shap_values(X_train_base,
                                nsamples=SHAP_NSAMPLES_GLOBAL, silent=True)
    return np.abs(np.asarray(sv)).mean(axis=0)

mean_abs_fo = headline_shap(X_tr_fo, y_tr_fo, 'first-order')
mean_abs_hx = headline_shap(X_tr_hx, y_tr_hx, 'Hixson')

# ============================================================================
# 3. Bootstrap stability — canonical run_b5
# ============================================================================
print("\n[bootstrap] Running run_b5 for first-order target ...")
b5_fo = run_b5(X_train_base=X_tr_fo, y_train_bc=y_tr_fo,
                feat_names=feat_names, n_iter=N_BOOT,
                target_name='first_order_rate', out_dir=OUT_DIR)
print("\n[bootstrap] Running run_b5 for Hixson target ...")
b5_hx = run_b5(X_train_base=X_tr_hx, y_train_bc=y_tr_hx,
                feat_names=feat_names, n_iter=N_BOOT,
                target_name='hixson_rate', out_dir=OUT_DIR)

top1_fo = {f: v * 100 for f, v in b5_fo['top1_freq'].items()}
top3_fo = {f: v * 100 for f, v in b5_fo['top3_freq'].items()}
top1_hx = {f: v * 100 for f, v in b5_hx['top1_freq'].items()}
top3_hx = {f: v * 100 for f, v in b5_hx['top3_freq'].items()}

# ============================================================================
# 4. Canonical ordering: Hixson top-3 frequency descending
#    Applied to ALL three rows (FO and Hixson within each row use the same order).
# ============================================================================
canonical_order = sorted(feat_names, key=lambda f: top3_hx[f])   # ascending — top of barh = top of canonical ordering

# Build per-target dicts of the headline SHAP indexed by feature name
shap_fo_dict = {f: mean_abs_fo[i] for i, f in enumerate(feat_names)}
shap_hx_dict = {f: mean_abs_hx[i] for i, f in enumerate(feat_names)}

# ============================================================================
# 5. Build the figure: 3 rows x 3 columns
# ============================================================================
fig = plt.figure(figsize=(15, 14))
fig.patch.set_facecolor('white')
gs = GridSpec(3, 3, figure=fig,
              width_ratios=[1.0, 1.0, 1.0],
              hspace=0.40, wspace=0.05,
              left=0.10, right=0.97, bottom=0.07, top=0.96)

def clean(ax):
    ax.grid(False); ax.set_facecolor('white')
    for s in ('top', 'right'): ax.spines[s].set_visible(False)
    for s in ('left', 'bottom'):
        ax.spines[s].set_color(TEXT_COLOR); ax.spines[s].set_linewidth(1.0)
    ax.tick_params(colors=TEXT_COLOR, which='both', labelsize=TICK_FS)
    for lbl in ax.get_xticklabels() + ax.get_yticklabels():
        lbl.set_color(TEXT_COLOR)

def draw_left_panel(ax, ordered_feats, value_dict, xlabel, fmt='{:.0f}%'):
    """Mirrored bar chart, spine on the LEFT edge (figure outer edge).
    Bars grow LEFTWARD from x=0 at the right edge (inner side, toward centre)."""
    vals = [value_dict.get(f, 0.0) for f in ordered_feats]
    colors = [feat_color(f) for f in ordered_feats]
    ypos = np.arange(len(ordered_feats))
    bars = ax.barh(ypos, [-v for v in vals], color=colors, height=0.65,
                   edgecolor='black', linewidth=0.5)
    vmax = max(vals) if max(vals) > 0 else 1.0
    for bar, v in zip(bars, vals):
        ax.text(-v - vmax * 0.02, bar.get_y() + bar.get_height()/2,
                fmt.format(v), va='center', ha='right',
                fontsize=VALUE_FS, color=TEXT_COLOR)
    ax.set_yticks(ypos); ax.set_yticklabels([])
    ax.tick_params(axis='y', left=False, right=False, length=0)
    ax.set_xlim(-vmax * 1.30, 0)
    # Display absolute values on the x-axis (mirrored data is negative)
    ax.xaxis.set_major_formatter(FuncFormatter(lambda x, pos: f'{abs(x):.2g}'
                                                 if vmax < 10 else f'{abs(x):.0f}'))
    ax.set_xlabel(xlabel, fontsize=LABEL_FS, color=TEXT_COLOR)
    clean(ax)

def draw_right_panel(ax, ordered_feats, value_dict, xlabel, fmt='{:.0f}%'):
    """Normal bar chart, spine on the RIGHT edge (figure outer edge).
    Bars grow RIGHTWARD from x=0 at the left edge (inner side, toward centre)."""
    vals = [value_dict.get(f, 0.0) for f in ordered_feats]
    colors = [feat_color(f) for f in ordered_feats]
    ypos = np.arange(len(ordered_feats))
    bars = ax.barh(ypos, vals, color=colors, height=0.65,
                   edgecolor='black', linewidth=0.5)
    vmax = max(vals) if max(vals) > 0 else 1.0
    for bar, v in zip(bars, vals):
        ax.text(v + vmax * 0.02, bar.get_y() + bar.get_height()/2,
                fmt.format(v), va='center', ha='left',
                fontsize=VALUE_FS, color=TEXT_COLOR)
    ax.set_yticks(ypos); ax.set_yticklabels([])
    ax.tick_params(axis='y', left=False, right=False, length=0)
    ax.set_xlim(0, vmax * 1.30)
    ax.set_xlabel(xlabel, fontsize=LABEL_FS, color=TEXT_COLOR)
    clean(ax)
    # Flip spine to right edge
    ax.spines['left'].set_visible(False)
    ax.spines['right'].set_visible(True)
    ax.spines['right'].set_color(TEXT_COLOR)
    ax.spines['right'].set_linewidth(1.0)

def draw_centre_labels(ax, ordered_feats):
    """Centre column: feature names only, no axes, no bars."""
    ypos = np.arange(len(ordered_feats))
    for y, f in zip(ypos, ordered_feats):
        ax.text(0.5, y, f, ha='center', va='center',
                fontsize=FEATNAME_FS, color=TEXT_COLOR, fontweight='normal',
                transform=ax.transData)
    ax.set_xlim(0, 1)
    ax.set_ylim(-0.5, len(ordered_feats) - 0.5)
    ax.set_xticks([]); ax.set_yticks([])
    ax.tick_params(axis='both', left=False, right=False, top=False, bottom=False,
                   length=0)
    for s in ('top', 'right', 'left', 'bottom'): ax.spines[s].set_visible(False)
    ax.set_facecolor('white')

# Row 0 — SHAP global
ax_a = fig.add_subplot(gs[0, 0])
ax_l0 = fig.add_subplot(gs[0, 1], sharey=ax_a)
ax_b = fig.add_subplot(gs[0, 2], sharey=ax_a)
draw_left_panel (ax_a,  canonical_order, shap_fo_dict, 'Mean $|$SHAP$|$ (first-order)', fmt='{:.3f}')
draw_centre_labels(ax_l0, canonical_order)
draw_right_panel(ax_b,  canonical_order, shap_hx_dict, 'Mean $|$SHAP$|$ (Hixson)',      fmt='{:.3f}')

# Row 1 — Top-3 set frequency
ax_c = fig.add_subplot(gs[1, 0])
ax_l1 = fig.add_subplot(gs[1, 1], sharey=ax_c)
ax_d = fig.add_subplot(gs[1, 2], sharey=ax_c)
draw_left_panel (ax_c,  canonical_order, top3_fo, 'Top-3 set frequency (%, first-order)')
draw_centre_labels(ax_l1, canonical_order)
draw_right_panel(ax_d,  canonical_order, top3_hx, 'Top-3 set frequency (%, Hixson)')

# Row 2 — Rank-1 frequency
ax_e = fig.add_subplot(gs[2, 0])
ax_l2 = fig.add_subplot(gs[2, 1], sharey=ax_e)
ax_f = fig.add_subplot(gs[2, 2], sharey=ax_e)
draw_left_panel (ax_e,  canonical_order, top1_fo, 'Rank-1 frequency (%, first-order)')
draw_centre_labels(ax_l2, canonical_order)
draw_right_panel(ax_f,  canonical_order, top1_hx, 'Rank-1 frequency (%, Hixson)')

# Row labels on the far left identifying each metric row
fig.text(0.022, 0.825, 'SHAP global',
         rotation=90, va='center', ha='center',
         fontsize=ROWLBL_FS, fontweight='normal', color=TEXT_COLOR)
fig.text(0.022, 0.515, 'Top-3 set frequency',
         rotation=90, va='center', ha='center',
         fontsize=ROWLBL_FS, fontweight='normal', color=TEXT_COLOR)
fig.text(0.022, 0.205, 'Rank-1 frequency',
         rotation=90, va='center', ha='center',
         fontsize=ROWLBL_FS, fontweight='normal', color=TEXT_COLOR)

# Shared legend at the bottom
legend_handles = [
    Patch(facecolor=MATRIX_BLUE, edgecolor='black', label='Matrix-derived'),
    Patch(facecolor=FIBRE_GREEN, edgecolor='black', label='Fibre-derived'),
]
fig.legend(handles=legend_handles, loc='lower center',
           bbox_to_anchor=(0.5, 0.005), ncol=2,
           frameon=True, edgecolor='black', fontsize=LEGEND_FS,
           labelcolor=TEXT_COLOR)

plt.savefig(OUT_PNG, dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig(OUT_PDF,            bbox_inches='tight', facecolor='white')
plt.show()
print(f"\n[output] Saved {OUT_PNG} and {OUT_PDF}")

# ============================================================================
# 6. Summary
# ============================================================================
print("\n" + "=" * 70)
print("SIX-PANEL TORNADO SUMMARY")
print("=" * 70)
print(f"\nCanonical ordering (Hixson top-3 frequency descending):")
sorted_canon = sorted(feat_names, key=lambda f: -top3_hx[f])
print(f"\n  {'feature':40s}  {'|SHAP|':>10s}  {'top-3 %':>10s}  {'rank-1 %':>10s}")
print(f"  {'(FO / Hix)':40s}  {'(FO / Hix)':>10s}  {'(FO / Hix)':>10s}  {'(FO / Hix)':>10s}")
for f in sorted_canon:
    print(f"  {f:40s}  "
          f"{shap_fo_dict[f]:.3f}/{shap_hx_dict[f]:.3f}  "
          f"{top3_fo[f]:>3.0f}%/{top3_hx[f]:>3.0f}%  "
          f"{top1_fo[f]:>3.0f}%/{top1_hx[f]:>3.0f}%")

## SHAP Beeswarm Plot

In [ ]:
# =============================================================================
# SHAP beeswarm tornado — back-to-back FO and Hixson, single composite figure
# =============================================================================
# Drop-in. Imports run_b5 / reproduce_pipeline from colored_analyses_b2_b6.py
# (same canonical pipeline as your bootstrap composite, so feature ordering
# matches that figure).
#
# Layout:
#
#     SHAP beeswarm (first-order)      Feature names         SHAP beeswarm (Hixson)
#     <----- mirrored, spine LEFT       (centre column)       normal, spine RIGHT ----->
#     ←··· • ••• •·                       logP_matrix          ·•••• • • ··· →
#     ←·· • •·                            tortuosity           ·•• •  · ··→
#     ...
#
# Mirroring detail:
#   - Each composite contributes one SHAP value per feature, plotted as a point.
#   - On the left panel, SHAP = +0.3 plots at x = -0.3 (positive points sit
#     LEFT of the centre, growing leftward away from the spine on the OUTER LEFT
#     edge). SHAP = -0.3 plots at x = +0.3 (negative points sit RIGHT of the
#     centre, growing toward the labels column).
#   - The x-axis tick labels on the left panel are FLIPPED in sign so the reader
#     reads "negative SHAP on the LEFT, positive SHAP on the RIGHT" — same
#     convention as the right panel. Only the spatial layout is mirrored.
#
# Colour: per-point RdBu_r based on feature value (low → blue, high → red).
# Centre column uses feature names colour-coded matrix-blue / fibre-green.
#
# Cost: ~3-5 min on real data (2 baseline SHAP computations at 150 nsamples).
# Panel labels and chrome left for you.
# =============================================================================

import warnings
warnings.filterwarnings('ignore')

import copy
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.gridspec import GridSpec
from matplotlib.colors import Normalize
from matplotlib.ticker import FuncFormatter
from sklearn.preprocessing import StandardScaler
import shap

from colored_analyses_b2_b6 import (
    reproduce_pipeline, run_b5,
    make_add_interactions, default_gpr_factory,
    MATRIX_BLUE, FIBRE_GREEN, FEATURE_CLASS,
)

# ---- Configuration ----------------------------------------------------------
DATASET_PATH = 'dataset.xlsx'
OUT_PNG      = 'figure_shap_beeswarm_tornado.png'
OUT_PDF      = 'figure_shap_beeswarm_tornado.pdf'
OUT_DIR      = './analyses_output'
N_BOOT       = 60                  # used only to derive canonical ordering
SHAP_NSAMPLES_GLOBAL = 150

# Typography — all text BLACK
TEXT_COLOR  = 'black'
TICK_FS     = 16
LABEL_FS    = 16
LEGEND_FS   = 14
FEATNAME_FS = 16
CBAR_FS     = 16

def feat_color(name: str) -> str:
    cls = FEATURE_CLASS.get(name, 'fibre')
    return MATRIX_BLUE if cls == 'matrix' else FIBRE_GREEN

# ============================================================================
# 1. Canonical pipeline + ordering (matches your bootstrap composite)
# ============================================================================
print("[setup] Reproducing canonical pipeline for both targets ...")
pipe_fo = reproduce_pipeline(DATASET_PATH, target='first_order_rate', verbose=True)
pipe_hx = reproduce_pipeline(DATASET_PATH, target='hixson_rate', verbose=True)

X_tr_fo, y_tr_fo = pipe_fo['X_train_base'], pipe_fo['y_train_bc']
X_tr_hx, y_tr_hx = pipe_hx['X_train_base'], pipe_hx['y_train_bc']
feat_names = pipe_fo['feat_names']

# Canonical ordering: Hixson top-3 frequency descending (same as bootstrap figure)
print("\n[ordering] Running run_b5 on Hixson to derive canonical ordering ...")
b5_hx = run_b5(X_train_base=X_tr_hx, y_train_bc=y_tr_hx,
                feat_names=feat_names, n_iter=N_BOOT,
                target_name='hixson_rate', out_dir=OUT_DIR)
top3_hx = {f: v for f, v in b5_hx['top3_freq'].items()}
canonical_order = sorted(feat_names, key=lambda f: top3_hx[f])   # asc — top of plot = top of canonical

# ============================================================================
# 2. SHAP values for both targets (same machinery as run_b5 internals)
# ============================================================================
def compute_shap(X_train_base, y_train_bc, label):
    add_int = make_add_interactions(feat_names)
    sc = StandardScaler().fit(add_int(X_train_base))
    gpr = copy.deepcopy(default_gpr_factory())
    gpr.fit(sc.transform(add_int(X_train_base)), y_train_bc)
    def f(X_query_base):
        return gpr.predict(sc.transform(add_int(X_query_base)))
    print(f"[shap] {label}: KernelSHAP, nsamples={SHAP_NSAMPLES_GLOBAL} ...")
    explainer = shap.KernelExplainer(
        f, X_train_base[:min(20, len(X_train_base))], silent=True)
    sv = explainer.shap_values(X_train_base,
                                nsamples=SHAP_NSAMPLES_GLOBAL, silent=True)
    return np.asarray(sv)

shap_fo = compute_shap(X_tr_fo, y_tr_fo, 'first-order')
shap_hx = compute_shap(X_tr_hx, y_tr_hx, 'Hixson')

# ============================================================================
# 3. Density-based jitter (so dense regions spread wider, sparse regions narrow)
# ============================================================================
def density_jitter(values, max_jitter=0.38, n_bins=20):
    values = np.asarray(values)
    if len(values) == 0:
        return values
    edges = np.linspace(values.min(), values.max() + 1e-9, n_bins + 1)
    bin_idx = np.clip(np.digitize(values, edges) - 1, 0, n_bins - 1)
    bin_counts = np.bincount(bin_idx, minlength=n_bins)
    offsets = np.zeros(len(values))
    for b in range(n_bins):
        in_bin = np.where(bin_idx == b)[0]
        n_in = len(in_bin)
        if n_in == 0: continue
        density = n_in / max(bin_counts.max(), 1)
        local_jitter = max_jitter * np.sqrt(density)
        if n_in == 1:
            offsets[in_bin] = 0.0
        else:
            offsets[in_bin] = np.linspace(-local_jitter, local_jitter, n_in)
            rng = np.random.RandomState(b * 17 + 3)
            offsets[in_bin] = rng.permutation(offsets[in_bin])
    return offsets

# ============================================================================
# 4. Build the figure: 1 row x 3 cols + colorbar
# ============================================================================
fig = plt.figure(figsize=(17, 7))
fig.patch.set_facecolor('white')

# 4-column grid: left beeswarm | centre labels (WIDE) | right beeswarm | colorbar
gs = GridSpec(1, 4, figure=fig,
              width_ratios=[3.0, 2.6, 3.0, 0.15],
              wspace=0.05,
              left=0.05, right=0.95, bottom=0.13, top=0.95)

# Compute a SHARED x-range so both panels are visually comparable
all_shap = np.concatenate([shap_fo.flatten(), shap_hx.flatten()])
xmax_abs = max(abs(all_shap.min()), abs(all_shap.max())) * 1.10

def clean(ax):
    ax.grid(False); ax.set_facecolor('white')
    ax.tick_params(colors=TEXT_COLOR, which='both', labelsize=TICK_FS)
    for lbl in ax.get_xticklabels() + ax.get_yticklabels():
        lbl.set_color(TEXT_COLOR)

def normalise_feature(values):
    """Robust 5-95 percentile normalisation for the colour scale."""
    p5, p95 = np.percentile(values, [5, 95])
    if p95 - p5 < 1e-9:
        return np.full_like(values, 0.5)
    return np.clip((values - p5) / (p95 - p5), 0, 1)

# ── Left panel (first-order, mirrored) ───────────────────────────────────────
ax_left = fig.add_subplot(gs[0, 0])
for rank, fname in enumerate(canonical_order):
    fi = feat_names.index(fname)
    sv = shap_fo[:, fi]
    fv = X_tr_fo[:, fi]
    fv_norm = normalise_feature(fv)
    point_colors = cm.RdBu_r(fv_norm)
    y_off = density_jitter(sv, max_jitter=0.38)
    # Mirror: plot SHAP value NEGATED so positive SHAP appears on the LEFT
    # side (away from centre labels). x-axis ticks will be flipped to match.
    ax_left.scatter(-sv, np.full(len(sv), rank) + y_off,
                     c=point_colors, s=34, alpha=0.85,
                     linewidths=0.3, edgecolors='black', zorder=3)

ax_left.axvline(0, color='black', lw=0.9, ls='--', zorder=2)
ax_left.set_xlim(-xmax_abs, xmax_abs)
# x-axis tick labels: data is plotted as -sv (mirrored) so axis is in flipped
# sign-space. Display labels as -x so the reader sees "negative SHAP on the
# outer-left, positive SHAP at the inner-right" — exactly the convention of
# the right panel, just spatially reflected.
ax_left.xaxis.set_major_formatter(FuncFormatter(lambda x, pos: f'{-x:.2f}'))
ax_left.set_xlabel('SHAP value (first-order)', fontsize=LABEL_FS, color=TEXT_COLOR)
ax_left.set_yticks(np.arange(len(canonical_order)))
ax_left.set_yticklabels([])
ax_left.tick_params(axis='y', left=False, right=False, length=0)
# Spine on the OUTER LEFT edge; hide top, right, bottom-extension-to-centre
for s in ('top', 'right'): ax_left.spines[s].set_visible(False)
ax_left.spines['left'].set_color(TEXT_COLOR); ax_left.spines['left'].set_linewidth(1.0)
ax_left.spines['bottom'].set_color(TEXT_COLOR); ax_left.spines['bottom'].set_linewidth(1.0)
clean(ax_left)

# ── Centre column: feature names ─────────────────────────────────────────────
ax_ctr = fig.add_subplot(gs[0, 1], sharey=ax_left)
for y, f in enumerate(canonical_order):
    ax_ctr.text(0.5, y, f, ha='center', va='center',
                 fontsize=FEATNAME_FS, color=TEXT_COLOR, fontweight='normal',
                 transform=ax_ctr.transData)
ax_ctr.set_xlim(0, 1)
ax_ctr.set_xticks([]); ax_ctr.set_yticks([])
ax_ctr.tick_params(axis='both', left=False, right=False, top=False,
                    bottom=False, length=0)
for s in ('top', 'right', 'left', 'bottom'): ax_ctr.spines[s].set_visible(False)
ax_ctr.set_facecolor('white')

# ── Right panel (Hixson, normal) ─────────────────────────────────────────────
ax_right = fig.add_subplot(gs[0, 2], sharey=ax_left)
for rank, fname in enumerate(canonical_order):
    fi = feat_names.index(fname)
    sv = shap_hx[:, fi]
    fv = X_tr_hx[:, fi]
    fv_norm = normalise_feature(fv)
    point_colors = cm.RdBu_r(fv_norm)
    y_off = density_jitter(sv, max_jitter=0.38)
    ax_right.scatter(sv, np.full(len(sv), rank) + y_off,
                      c=point_colors, s=34, alpha=0.85,
                      linewidths=0.3, edgecolors='black', zorder=3)

ax_right.axvline(0, color='black', lw=0.9, ls='--', zorder=2)
ax_right.set_xlim(-xmax_abs, xmax_abs)
ax_right.xaxis.set_major_formatter(FuncFormatter(lambda x, pos: f'{x:.2f}'))
ax_right.set_xlabel('SHAP value (Hixson)', fontsize=LABEL_FS, color=TEXT_COLOR)
ax_right.set_yticks(np.arange(len(canonical_order)))
ax_right.set_yticklabels([])
ax_right.tick_params(axis='y', left=False, right=False, length=0)
for s in ('top', 'left'): ax_right.spines[s].set_visible(False)
ax_right.spines['right'].set_color(TEXT_COLOR); ax_right.spines['right'].set_linewidth(1.0)
ax_right.spines['bottom'].set_color(TEXT_COLOR); ax_right.spines['bottom'].set_linewidth(1.0)
clean(ax_right)

# ── Colorbar ─────────────────────────────────────────────────────────────────
cax = fig.add_subplot(gs[0, 3])
sm = cm.ScalarMappable(cmap='RdBu_r', norm=Normalize(vmin=0, vmax=1))
sm.set_array([])
cbar = fig.colorbar(sm, cax=cax)
cbar.set_label('Feature value (low → high)', fontsize=CBAR_FS,
                rotation=270, labelpad=22, color=TEXT_COLOR)
cbar.set_ticks([0.0, 0.5, 1.0])
cbar.set_ticklabels(['low', 'median', 'high'])
cbar.ax.tick_params(labelsize=CBAR_FS, colors=TEXT_COLOR)
cbar.outline.set_edgecolor('black')
cbar.outline.set_linewidth(0.6)

plt.savefig(OUT_PNG, dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig(OUT_PDF,            bbox_inches='tight', facecolor='white')
plt.show()
print(f"\n[output] Saved {OUT_PNG} and {OUT_PDF}")

# ============================================================================
# 5. Summary
# ============================================================================
print("\n" + "=" * 70)
print("SHAP TORNADO SUMMARY — canonical ordering (Hixson top-3 desc)")
print("=" * 70)
mean_abs_fo = np.abs(shap_fo).mean(axis=0)
mean_abs_hx = np.abs(shap_hx).mean(axis=0)
print(f"\n  {'feature':40s}  {'|SHAP| FO':>10s}  {'|SHAP| Hix':>10s}  {'top-3 Hix':>10s}")
for f in canonical_order[::-1]:
    fi = feat_names.index(f)
    print(f"  {f:40s}  {mean_abs_fo[fi]:>10.4f}  "
          f"{mean_abs_hx[fi]:>10.4f}  {top3_hx[f]*100:>9.0f}%")

Panelled figure for log transformed sensitivity

In [ ]:
# =============================================================================
# Combined log-transform sensitivity — two stacked panels (b6 logic, both targets)
# =============================================================================
# Drop-in. Imports run_b6 from colored_analyses_b2_b6.py so the numbers match
# the manuscript exactly.
#
# Each panel runs run_b6's exact comparison (SHAP mean |value| WITH vs WITHOUT
# the log-transform on the five right-skewed features) for one target:
#
#   Panel a (top):    first-order rate    — solid bar = with-log,
#                                            hatched bar = without-log
#   Panel b (bottom): Hixson-Crowell rate — same encoding
#
# Both panels share the SAME feature ordering and the SAME x-axis, so a feature's
# log-sensitivity can be read vertically across the two targets. Ordering is by
# Hixson's (with_log + without_log) sum descending — consistent with the
# Hixson-ordered convention used in the other composite figures.
#
# Cost: ~6-10 min on real data (run_b6 computes 2 SHAP passes per target,
# 4 total).
# Panel labels and chrome left for you.
# =============================================================================

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Patch

from colored_analyses_b2_b6 import (
    reproduce_pipeline, run_b6,
    MATRIX_BLUE, FIBRE_GREEN, FEATURE_CLASS,
)

# ---- Configuration ----------------------------------------------------------
DATASET_PATH = 'dataset.xlsx'
OUT_PNG      = 'figure_log_sensitivity_combined.png'
OUT_PDF      = 'figure_log_sensitivity_combined.pdf'
OUT_DIR      = './analyses_output'

# Typography — all text BLACK
TEXT_COLOR = 'black'
TICK_FS    = 12
LABEL_FS   = 14
LEGEND_FS  = 12
ROWLBL_FS  = 15

def feat_color(name: str) -> str:
    cls = FEATURE_CLASS.get(name, 'fibre')
    return MATRIX_BLUE if cls == 'matrix' else FIBRE_GREEN

# ============================================================================
# 1. Canonical pipeline for both targets
# ============================================================================
print("[setup] Reproducing canonical pipeline for first-order target ...")
pipe_fo = reproduce_pipeline(DATASET_PATH, target='first_order_rate', verbose=True)
print("\n[setup] Reproducing canonical pipeline for Hixson target ...")
pipe_hx = reproduce_pipeline(DATASET_PATH, target='hixson_rate', verbose=True)

feat_names = pipe_fo['feat_names']

# ============================================================================
# 2. Run b6 for both targets
# ----------------------------------------------------------------------------
# run_b6 needs: X_train_log (the headline log-transformed matrix = X_train_base),
# X_train_raw (the un-logged version = X_train_raw_for_b6), and y_train_bc.
# ============================================================================
print("\n[b6] Running log-sensitivity analysis for first-order target ...")
b6_fo = run_b6(X_train_log=pipe_fo['X_train_base'],
               X_train_raw=pipe_fo['X_train_raw_for_b6'],
               y_train_bc=pipe_fo['y_train_bc'],
               feat_names=feat_names,
               target_name='first_order_rate', out_dir=OUT_DIR)

print("\n[b6] Running log-sensitivity analysis for Hixson target ...")
b6_hx = run_b6(X_train_log=pipe_hx['X_train_base'],
               X_train_raw=pipe_hx['X_train_raw_for_b6'],
               y_train_bc=pipe_hx['y_train_bc'],
               feat_names=feat_names,
               target_name='hixson_rate', out_dir=OUT_DIR)

with_fo, no_fo = b6_fo['shap_with_log'], b6_fo['shap_without_log']
with_hx, no_hx = b6_hx['shap_with_log'], b6_hx['shap_without_log']

# ============================================================================
# 3. Shared feature ordering — Hixson (with_log + without_log) sum, descending
# ============================================================================
order = sorted(feat_names, key=lambda f: -(with_hx[f] + no_hx[f]))

# ============================================================================
# 4. Build the two-panel figure
# ============================================================================
fig = plt.figure(figsize=(13, 11))
fig.patch.set_facecolor('white')
gs = GridSpec(2, 1, figure=fig, hspace=0.30,
              left=0.13, right=0.97, bottom=0.20, top=0.95)

W = 0.38   # bar width

def draw_b6_panel(ax, with_log, no_log, xlabel_show):
    """Replicates run_b6's grouped bar chart: solid = with-log, hatched = without."""
    x = np.arange(len(order))
    colors = [feat_color(f) for f in order]

    # With-log: solid fill in the feature's class colour
    ax.bar(x - W/2, [with_log[f] for f in order], W,
           color=colors, edgecolor='black', linewidth=0.7)

    # Without-log: white fill, hatched, with a coloured rim to keep feature identity
    ax.bar(x + W/2, [no_log[f] for f in order], W,
           color='white', edgecolor='black', linewidth=0.7, hatch='///')
    for xi, f in zip(x, order):
        ax.bar(xi + W/2, no_log[f], W,
               color='none', edgecolor=feat_color(f), linewidth=1.6)

    ax.set_xticks(x)
    ax.set_xticklabels(order, rotation=30, ha='right',
                       fontsize=TICK_FS, color=TEXT_COLOR)
    if not xlabel_show:
        # Suppress x-tick labels on this axis without affecting the sharex
        # partner (set_xticklabels([]) would propagate through sharex).
        ax.tick_params(axis='x', labelbottom=False)
    ax.set_ylabel(r'Mean $|$SHAP$|$', fontsize=LABEL_FS, color=TEXT_COLOR)

    # Clean style
    ax.grid(False); ax.set_facecolor('white')
    for s in ('top', 'right'): ax.spines[s].set_visible(False)
    for s in ('left', 'bottom'):
        ax.spines[s].set_color(TEXT_COLOR); ax.spines[s].set_linewidth(1.0)
    ax.tick_params(colors=TEXT_COLOR, which='both', labelsize=TICK_FS)
    for lbl in ax.get_xticklabels() + ax.get_yticklabels():
        lbl.set_color(TEXT_COLOR)

# Panel a — first-order  (no x tick labels; shared with bottom panel)
ax_a = fig.add_subplot(gs[0, 0], sharex=ax_a)
draw_b6_panel(ax_a, with_fo, no_fo, xlabel_show=False)

# Panel b — Hixson  (x tick labels shown here)
ax_b = fig.add_subplot(gs[1, 0], sharex=ax_a)
draw_b6_panel(ax_b, with_hx, no_hx, xlabel_show=True)

# Row labels on the far left
fig.text(0.035, 0.72, 'First-order rate constant',
         rotation=90, va='center', ha='center',
         fontsize=ROWLBL_FS, fontweight='bold', color=TEXT_COLOR)
fig.text(0.035, 0.38, 'Hixson–Crowell rate constant',
         rotation=90, va='center', ha='center',
         fontsize=ROWLBL_FS, fontweight='bold', color=TEXT_COLOR)

# Shared legend at the bottom
legend_handles = [
    Patch(facecolor='lightgrey', edgecolor='black',
          label='With log transform (headline)'),
    Patch(facecolor='white', edgecolor='black', hatch='///',
          label='Without log transform'),
    Patch(facecolor=MATRIX_BLUE, edgecolor='black', label='Matrix-derived'),
    Patch(facecolor=FIBRE_GREEN, edgecolor='black', label='Fibre-derived'),
]
fig.legend(handles=legend_handles, loc='upper right',
           bbox_to_anchor=(0.5, 0.02), ncol=4,
           frameon=True, edgecolor='black', fontsize=LEGEND_FS,
           labelcolor=TEXT_COLOR)

plt.savefig(OUT_PNG, dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig(OUT_PDF,            bbox_inches='tight', facecolor='white')
plt.show()
print(f"\n[output] Saved {OUT_PNG} and {OUT_PDF}")

# ============================================================================
# 5. Summary
# ============================================================================
print("\n" + "=" * 70)
print("LOG-TRANSFORM SENSITIVITY SUMMARY (canonical run_b6 output)")
print("=" * 70)
for label, wl, nl, b6 in [
    ('First-order',    with_fo, no_fo, b6_fo),
    ('Hixson-Crowell', with_hx, no_hx, b6_hx),
]:
    print(f"\n{label}:")
    print(f"  {'feature':40s}  {'with-log':>10s}  {'without-log':>12s}")
    for f in order:
        print(f"  {f:40s}  {wl[f]:>10.3f}  {nl[f]:>12.3f}")
    print(f"  Top-3 with log:    {b6['top3_with_log']}")
    print(f"  Top-3 without log: {b6['top3_without_log']}")

For supplementary materials- which samples names inside AD?

In [ ]:
# =============================================================================
# Composite categorisation — outliers / outside AD / inside AD
# =============================================================================
# Drop-in. Indexes every composite by its ORIGINAL dataset row (0-61) and
# sorts each into one of three categories.
#
# WHY THIS CELL EXISTS
#   reproduce_pipeline() calls reset_index(drop=True) twice and never carries
#   the original dataset row numbers, so its output arrays cannot be mapped
#   back to dataset.xlsx rows. This cell REPLAYS the identical pipeline steps
#   (same dropna, same _PRELIM_OUTLIER_ROWS, same _kennard_stone, same worst-9
#   rule, same RANDOM_SEED) while carrying an `orig_idx` column through every
#   filter — so every composite keeps its original row number.
#
# CATEGORIES (as specified)
#   OUTLIERS      = the 3 preliminary outliers (dropped pre-modelling)
#                 + the 9 worst-LOO-residual composites (dropped from the
#                   training pool)
#                 + any RETAINED composite whose Williams standardised
#                   residual exceeds +/-3
#   OUTSIDE AD    = retained composites with Williams leverage h > h*
#                   (and not already an outlier)
#   INSIDE AD     = retained composites with h <= h* and |std residual| <= 3
#
#   AD verdicts use the Williams-plot leverage method, (p+1) intercept
#   convention, computed on the 8 base descriptors.
#
# IMPORTANT — NA ROWS
#   The pipeline's dropna step removes rows with missing feature values. If
#   dataset.xlsx has NO missing feature values, original row numbers are
#   preserved exactly. If it DOES have missing values, the cell still tracks
#   indices correctly, but be aware the preliminary-outlier rows [0,8,18] are
#   defined on the POST-dropna table (as in the pipeline) — the cell reports
#   both the post-dropna position and the original row for those.
# =============================================================================

import warnings
warnings.filterwarnings('ignore')

import copy
import numpy as np
import pandas as pd
from scipy.stats import boxcox
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneOut
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel as C, RBF, WhiteKernel)

from colored_analyses_b2_b6 import (
    _kennard_stone, make_add_interactions, default_gpr_factory,
    bc_to_log,
    _PRELIM_OUTLIER_ROWS, _N_EXT, _N_OUTLIERS, _COLLINEAR_DROP_DEFAULT,
    LOG_TRANSFORM_FEATS, RANDOM_SEED,
)

# ---- Configuration ----------------------------------------------------------
DATASET_PATH = 'dataset.xlsx'
TARGET_FO, TARGET_HIX = 'first_order_rate', 'hixson_rate'

# ============================================================================
# 1. Replay the pipeline, carrying orig_idx through every step
# ============================================================================
df0 = pd.read_excel(DATASET_PATH)
n_original = len(df0)
print(f"[setup] dataset.xlsx has {n_original} rows (original indices 0..{n_original-1})")

# orig_idx column = the original dataset row number
df0 = df0.copy()
df0['__orig_idx__'] = np.arange(n_original)

all_feats = [c for c in df0.columns
             if c not in (TARGET_FO, TARGET_HIX, '__orig_idx__')]

# --- Step 2: dropna across feature columns ---------------------------------
df = df0.dropna(subset=all_feats).reset_index(drop=True)
n_after_na = len(df)
n_na_dropped = n_original - n_after_na
if n_na_dropped:
    print(f"[setup] dropna removed {n_na_dropped} row(s) with missing features")
else:
    print(f"[setup] dropna removed 0 rows — original numbering preserved")

# --- Step 3: log-transform (features only; does not affect rows) -----------
for f in LOG_TRANSFORM_FEATS:
    if f in df.columns and (df[f].values > 0).all():
        df[f] = np.log(df[f].values)

# --- Step 4: drop collinear features (columns, not rows) -------------------
df.drop(columns=[c for c in _COLLINEAR_DROP_DEFAULT if c in df.columns],
        inplace=True)
feats_model = [f for f in all_feats if f in df.columns]

# --- Step 5: drop preliminary outliers (positions in the cleaned table) ----
prelim_orig = df.iloc[_PRELIM_OUTLIER_ROWS]['__orig_idx__'].tolist()
mask = np.ones(len(df), dtype=bool)
mask[_PRELIM_OUTLIER_ROWS] = False
df = df[mask].reset_index(drop=True)
print(f"[setup] preliminary outliers: cleaned-table rows {_PRELIM_OUTLIER_ROWS} "
      f"-> original rows {prelim_orig}")

# --- Step 6: Kennard-Stone split -------------------------------------------
orig_idx_all = df['__orig_idx__'].values.copy()
X_all = df[feats_model].values.astype(float)
raw_yfo  = df[TARGET_FO].values
raw_yhix = df[TARGET_HIX].values
log_yfo  = np.log(raw_yfo)

sc_ks = StandardScaler()
ks_idx = _kennard_stone(sc_ks.fit_transform(X_all), n_select=_N_EXT)
ks_mask = np.zeros(len(X_all), dtype=bool)
ks_mask[ks_idx] = True

X_train_pool = X_all[~ks_mask]
X_test_full  = X_all[ks_mask]
orig_pool = orig_idx_all[~ks_mask]
orig_ext  = orig_idx_all[ks_mask]
raw_yfo_pool = raw_yfo[~ks_mask]
log_yfo_pool = log_yfo[~ks_mask]

# --- Step 7: identify the 9 worst-LOO-residual outliers in the train pool --
bc_yfo_pool, lam_fo_pool = boxcox(raw_yfo_pool)
add_int = make_add_interactions(feats_model)
gpr_id = GaussianProcessRegressor(
    kernel=C(1.0, (1e-2, 1e2)) * RBF(1.0, (0.5, 20.0))
            + WhiteKernel(0.1, (1e-2, 1.0)),
    alpha=0.1, n_restarts_optimizer=5, normalize_y=True,
    random_state=RANDOM_SEED)
preds_id_bc = np.zeros(len(log_yfo_pool))
for tr, te in LeaveOneOut().split(X_train_pool):
    Xtr = add_int(X_train_pool[tr]); Xte = add_int(X_train_pool[te])
    sc = StandardScaler()
    m = copy.deepcopy(gpr_id)
    m.fit(sc.fit_transform(Xtr), bc_yfo_pool[tr])
    preds_id_bc[te] = m.predict(sc.transform(Xte)).ravel()[0]
preds_id_log = bc_to_log(preds_id_bc, lam_fo_pool)
residuals = log_yfo_pool - preds_id_log
std_res = residuals / residuals.std()
outlier_pool_idx = np.argsort(np.abs(std_res))[-_N_OUTLIERS:].tolist()
worst9_orig = orig_pool[outlier_pool_idx].tolist()

keep = np.ones(len(X_train_pool), dtype=bool)
keep[outlier_pool_idx] = False
X_train  = X_train_pool[keep]
orig_train = orig_pool[keep]
print(f"[setup] worst-9 LOO outliers -> original rows {sorted(worst9_orig)}")

# ============================================================================
# 2. Williams leverage on the retained composites (40 train + 9 external)
# ============================================================================
n_tr = len(X_train)
p = X_train.shape[1]

sc = StandardScaler().fit(X_train)
Xs_tr  = sc.transform(X_train)
Xs_ext = sc.transform(X_test_full)

D_tr  = np.column_stack([np.ones(n_tr),          Xs_tr])
D_ext = np.column_stack([np.ones(len(Xs_ext)),   Xs_ext])
p_eff = D_tr.shape[1]
DtD_inv = np.linalg.pinv(D_tr.T @ D_tr)
h_train = np.einsum('ij,jk,ik->i', D_tr,  DtD_inv, D_tr)
h_ext   = np.einsum('ij,jk,ik->i', D_ext, DtD_inv, D_ext)
h_star  = 3.0 * p_eff / n_tr

assert h_train.max() <= 1.0 + 1e-8, "training leverage > 1 — calculation wrong"

# Standardised residuals (LOO) for the retained training set + external
raw_yhix = df[TARGET_HIX].values
raw_yhix_pool = raw_yhix[~ks_mask]
raw_yhix_tr   = raw_yhix_pool[keep]
log_yhix = np.log(raw_yhix)
log_yhix_tr = log_yhix[~ks_mask][keep]
log_yhix_te = log_yhix[ks_mask]

bc_yhix_tr, lam_hix = boxcox(raw_yhix_tr)
sc_aug = StandardScaler().fit(add_int(X_train))
Xs_aug_tr = sc_aug.transform(add_int(X_train))
gpr_full = default_gpr_factory(n_restarts=10)
gpr_full.fit(Xs_aug_tr, bc_yhix_tr)
fitted_kernel = gpr_full.kernel_

loo_pred_bc = np.zeros(n_tr)
for tr, te in LeaveOneOut().split(Xs_aug_tr):
    g = GaussianProcessRegressor(kernel=fitted_kernel, alpha=gpr_full.alpha,
                                  normalize_y=True, optimizer=None)
    g.fit(Xs_aug_tr[tr], bc_yhix_tr[tr])
    loo_pred_bc[te[0]] = g.predict(Xs_aug_tr[te])[0]
loo_pred_log = bc_to_log(loo_pred_bc, lam_hix)
resid_tr = log_yhix_tr - loo_pred_log
sd_resid = resid_tr.std()
std_resid_tr = resid_tr / sd_resid

ext_pred_bc = gpr_full.predict(sc_aug.transform(add_int(X_test_full)))
ext_pred_log = bc_to_log(ext_pred_bc, lam_hix)
std_resid_ext = (log_yhix_te - ext_pred_log) / sd_resid

# ============================================================================
# 3. Categorise every composite
# ============================================================================
# OUTLIERS: 3 preliminary + 9 worst-LOO + retained composites with |resid| > 3
outliers = set(prelim_orig) | set(worst9_orig)

inside_ad  = []
outside_ad = []
resid_outliers_retained = []

for k in range(n_tr):
    oi = int(orig_train[k])
    if abs(std_resid_tr[k]) > 3.0:
        resid_outliers_retained.append(oi)
        outliers.add(oi)
    elif h_train[k] > h_star:
        outside_ad.append(oi)
    else:
        inside_ad.append(oi)

for k in range(len(X_test_full)):
    oi = int(orig_ext[k])
    if abs(std_resid_ext[k]) > 3.0:
        resid_outliers_retained.append(oi)
        outliers.add(oi)
    elif h_ext[k] > h_star:
        outside_ad.append(oi)
    else:
        inside_ad.append(oi)

outliers      = sorted(outliers)
outside_ad    = sorted(outside_ad)
inside_ad     = sorted(inside_ad)

# ============================================================================
# 4. Report
# ============================================================================
def fmt(lst):
    return ', '.join(str(i) for i in lst) if lst else '(none)'

print("\n" + "=" * 72)
print("COMPOSITE CATEGORISATION  (original dataset row indices)")
print("=" * 72)
print(f"  h* = 3(p+1)/n = {h_star:.4f}   |   residual outlier cutoff = +/-3 sigma")

print(f"\nSamples {fmt(outliers)} are OUTLIERS")
print(f"   - preliminary outliers (dropped pre-modelling) : {fmt(sorted(prelim_orig))}")
print(f"   - worst-9 LOO-residual (dropped from pool)      : {fmt(sorted(worst9_orig))}")
print(f"   - retained but |std. residual| > 3             : "
      f"{fmt(sorted(resid_outliers_retained))}")

print(f"\nSamples {fmt(outside_ad)} are OUTSIDE the applicability domain")
print(f"   (retained composites with leverage h > h* = {h_star:.4f})")

print(f"\nSamples {fmt(inside_ad)} are INSIDE the applicability domain")
print(f"   (retained composites with h <= h* and |std. residual| <= 3)")

# Integrity check — every original row accounted for exactly once
all_categorised = set(outliers) | set(outside_ad) | set(inside_ad)
expected = set(range(n_original))
missing  = sorted(expected - all_categorised)
dup_ok   = (len(outliers) + len(outside_ad) + len(inside_ad)
            == len(all_categorised))
print("\n" + "-" * 72)
print(f"Counts : {len(outliers)} outliers, {len(outside_ad)} outside AD, "
      f"{len(inside_ad)} inside AD  (total {len(all_categorised)}/{n_original})")
if missing:
    print(f"WARNING: original rows not categorised: {missing}")
    print("  (these were dropped by the dropna step — they have missing")
    print("   feature values and never entered the pipeline.)")
if not dup_ok:
    print("WARNING: a composite appears in more than one category — check logic.")

In-domain consistency figure

In [ ]:
# =============================================================================
# In-domain consistency figure — matrix chemistry vs degradation rate
# =============================================================================
# Shows that, RESTRICTED TO THE 43 COMPOSITES INSIDE THE APPLICABILITY DOMAIN,
# the matrix-chemistry contrast central to the full-dataset analysis is
# preserved: petroleum-derived matrices degrade an order of magnitude more
# slowly than biodegradable matrices.

# DATA
#   Reads the applicability-domain CSV directly (the file you provided).
#   Only the 43 composites flagged "Yes" in the "Inside AD?" column are plotted.
#   Every composite is shown as an individual point — no boxplots over the data.
#
# Self-contained: needs only pandas, numpy, matplotlib.
# =============================================================================

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---- Configuration ----------------------------------------------------------
CSV_PATH = 'applicability_domain.csv'   # <-- set to your CSV file path
OUT_PNG  = 'in_domain_matrix_chemistry.png'
OUT_PDF  = 'in_domain_matrix_chemistry.pdf'

# Colours (manuscript scheme)
PETRO_COLOR = '#185FA5'     # petroleum-derived matrices
BIO_COLOR   = '#3B6D11'     # biodegradable matrices
MEDIAN_COL  = '#E24B4A'
TEXT_COLOR  = 'black'

# Typography — large
TICK_FS   = 15
LABEL_FS  = 18
TITLE_FS  = 19
SUB_FS    = 13
ANNOT_FS  = 14

# ============================================================================
# 1. Load the CSV and select inside-AD composites
# ============================================================================
df = pd.read_csv(CSV_PATH)
df.columns = [c.strip() for c in df.columns]

ad_col = [c for c in df.columns if 'Inside AD' in c][0]
rate_col = [c for c in df.columns if 'First Order' in c][0]

inside = df[df[ad_col].astype(str).str.strip().str.startswith('Yes')].copy()
print(f"[data] {len(inside)} composites inside the applicability domain "
      f"(of {len(df)} total)")

# ============================================================================
# 2. Classify matrix chemistry — petroleum-derived vs biodegradable
# ============================================================================
# Petroleum-derived: PP, HDPE, and epoxy-based matrices.
# Biodegradable    : PLA, TPS, PCL, PBS / Bio-PBS / PBSu, PHA, PHBV,
#                    poly(3HB-co-3HHx), starch-, wood-flour-based, etc.
PETRO_KEYS = ['pp', 'hdpe', 'epoxy']

def matrix_class(name):
    n = str(name).lower()
    # 'bio-pbs' must not be caught by a petro key; check petro keys as words
    if any(n == k or n.startswith(k + ' ') or n.startswith(k + '/')
           or n == k.upper().lower() for k in PETRO_KEYS):
        return 'petroleum'
    # explicit catch for the plain petro matrices and epoxy variants
    if n in ('pp', 'hdpe') or n.startswith('epoxy'):
        return 'petroleum'
    return 'biodegradable'

inside['mclass'] = inside['Matrix'].apply(matrix_class)

petro = inside[inside['mclass'] == 'petroleum']
bio   = inside[inside['mclass'] == 'biodegradable']
print(f"[data] petroleum-derived: n = {len(petro)}")
print(f"[data] biodegradable    : n = {len(bio)}")

# ============================================================================
# 3. Strip / jitter plot
# ============================================================================
fig, ax = plt.subplots(figsize=(11, 7.5))
fig.patch.set_facecolor('white')

groups = [
    ('Petroleum-derived\n(PP, HDPE, epoxy)', petro, PETRO_COLOR, 0),
    ('Biodegradable\n(PLA, TPS, PCL, PBS, PHA)', bio,  BIO_COLOR,  1),
]

rng = np.random.RandomState(7)
for label, g, color, xc in groups:
    rates = g[rate_col].astype(float).values
    # horizontal jitter so overlapping points are visible
    jit = rng.uniform(-0.18, 0.18, size=len(rates))
    ax.scatter(np.full(len(rates), xc) + jit, rates,
               s=130, color=color, alpha=0.80,
               edgecolors='black', linewidths=0.8, zorder=3)
    # median line
    med = np.median(rates)
    ax.plot([xc - 0.32, xc + 0.32], [med, med],
            color=MEDIAN_COL, lw=3.0, zorder=4, solid_capstyle='round')
    # median value annotation
    ax.text(xc + 0.40, med, f'median\n{med:.4f}',
            ha='left', va='center', fontsize=ANNOT_FS,
            fontweight='bold', color=MEDIAN_COL)
    # n label under the group
    ax.text(xc, ax.get_ylim()[0], '', )   # placeholder; n added after ylim set

ax.set_yscale('log')
ax.set_xlim(-0.6, 1.75)
ax.set_xticks([0, 1])
ax.set_xticklabels([g[0] for g in groups], fontsize=TICK_FS, color=TEXT_COLOR)
ax.set_ylabel('First-order degradation rate constant  (log scale)',
              fontsize=LABEL_FS, color=TEXT_COLOR)

# n labels just above the x tick labels
ymin, ymax = ax.get_ylim()
for label, g, color, xc in groups:
    ax.text(xc, ymin * 1.4, f'n = {len(g)}',
            ha='center', va='bottom', fontsize=ANNOT_FS,
            fontweight='bold', color=color)

ax.set_title('Matrix chemistry within the applicability domain',
             fontsize=TITLE_FS, fontweight='bold', color=TEXT_COLOR, pad=32)
ax.text(0.5, 1.015,
        'Each point is one in-domain composite. Petroleum matrices cluster '
        'low; biodegradable matrices span a wide range, reaching far faster.',
        transform=ax.transAxes, ha='center', va='bottom',
        fontsize=SUB_FS, style='italic', color='#444444')

# Fold-difference annotation between the two medians
med_p = np.median(petro[rate_col].astype(float).values)
med_b = np.median(bio[rate_col].astype(float).values)
fold = med_b / med_p
ax.annotate('', xy=(0.5, med_b), xytext=(0.5, med_p),
            arrowprops=dict(arrowstyle='<->', color='#444444', lw=1.6))
ax.text(0.56, np.sqrt(med_p * med_b),
        f'{fold:.0f}x\n(medians)', fontsize=ANNOT_FS, fontweight='bold',
        color='#444444', ha='left', va='center')

ax.grid(False); ax.set_facecolor('white')
for s in ('top', 'right'): ax.spines[s].set_visible(False)
for s in ('left', 'bottom'):
    ax.spines[s].set_color('black'); ax.spines[s].set_linewidth(1.2)
ax.tick_params(colors=TEXT_COLOR, which='both', labelsize=TICK_FS)
for lbl in ax.get_xticklabels() + ax.get_yticklabels():
    lbl.set_color(TEXT_COLOR)

plt.tight_layout()
plt.savefig(OUT_PNG, dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig(OUT_PDF,            bbox_inches='tight', facecolor='white')
plt.show()
print(f"\n[output] Saved {OUT_PNG} and {OUT_PDF}")

# ============================================================================
# 4. Numerical summary
# ============================================================================
print("\n" + "=" * 64)
print("IN-DOMAIN MATRIX-CHEMISTRY SUMMARY (first-order rate)")
print("=" * 64)
for label, g, _, _ in groups:
    r = g[rate_col].astype(float)
    tag = label.split('\n')[0]
    print(f"\n{tag}  (n = {len(g)}):")
    print(f"  median {r.median():.5f}   mean {r.mean():.5f}")
    print(f"  range  {r.min():.5f} - {r.max():.5f}")
print(f"\nMedian fold-difference (biodegradable / petroleum): {fold:.1f}x")
max_fold = bio[rate_col].astype(float).max() / med_p
print(f"Max fold-difference (fastest biodegradable / petroleum median): "
      f"{max_fold:.0f}x")

## Logic of N_outliers = 9

In [ ]:
import pandas as pd
import numpy as np
import copy
import warnings
import os

from scipy.stats import spearmanr, pearsonr, boxcox
from scipy.spatial.distance import cdist
from sklearn.preprocessing import StandardScaler
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C, WhiteKernel
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import r2_score

warnings.filterwarnings('ignore')

# ==============================================================================
#  CONFIGURATION
# ==============================================================================
DATA_PATH       = 'dataset.xlsx'
TARGET_FO       = 'first_order_rate'
TARGET_HIX      = 'hixson_rate'
OUTPUT_DIR      = '.'
RANDOM_SEED     = 42
N_PERM          = 100
N_RESTARTS      = 10
# N_OUTLIERS will be looped from 5 to 15

LOG_TRANSFORM_FEATS = [
    'Half life_matrix_avg', 'tortuosity', 'interface_rot',
    'matrix_CED', 'fiber_dia_or_average_particle_size_mm',
]
COLLINEAR_DROP = [
    'hydrophyllic_gradient', 'matrix_sol_parameter', 'flory_huggins',
    'logKOC_matrix_avg', 'sol_mismatch', 'logP_fiber', 'Lignin_Percentage',
    'Cellulose_Percentage', 'Half life_fiber_avg', 'logKOC_fiber_avg',
    'fiber_CED', 'fiber_rotatable_bond_count', 'Specific Interface Area',
]
PRELIM_OUTLIER_ROWS = [0, 8, 18]

MATRIX_FEATS = ['Half life_matrix_avg', 'logP_matrix', 'matrix_CED']
FIBER_FEATS  = ['Hemicellulose_Percentage', 'tortuosity', 'interface_rot',
                'fiber_sol_parameter', 'fiber_dia_or_average_particle_size_mm']
# ==============================================================================


def banner(text):
    print(f"\n{'='*70}")
    print(f"  {text}")
    print(f"{'='*70}")

def kennard_stone(X, n_select):
    dist = cdist(X, X, 'euclidean')
    i1, i2 = np.unravel_index(np.argmax(dist), dist.shape)
    sel = [i1, i2]; rem = list(set(range(len(X))) - set(sel))
    while len(sel) < n_select:
        d = cdist(X[rem], X[sel], 'euclidean')
        best = np.argmax(np.min(d, axis=1))
        sel.append(rem[best]); rem.pop(best)
    return np.array(sel)

def add_interactions(X):
    return np.hstack([X] + [X[:, m:m+1] * X[:, f:f+1]
                             for m in MATRIX_IDX for f in FIBER_IDX])

def bc_to_log(p, lam, eps=1e-10):
    if abs(lam) < 0.01: return p
    return np.log(np.clip(np.power(np.clip(p*lam+1, eps, None), 1.0/lam), eps, None))

def loo_predictions(X, y_bc, lam, model):
    preds_bc = np.zeros(len(y_bc))
    for tr, te in LeaveOneOut().split(X):
        Xtr = add_interactions(X[tr])
        Xte = add_interactions(X[te])
        sc = StandardScaler(); m = copy.deepcopy(model)
        m.fit(sc.fit_transform(Xtr), y_bc[tr])
        preds_bc[te] = m.predict(sc.transform(Xte)).ravel()[0]
    return bc_to_log(preds_bc, lam)


def fit_and_predict(X_tr, y_bc, X_te, lam, model):
    sc = StandardScaler(); m = copy.deepcopy(model)
    X_tr_aug = add_interactions(X_tr); X_te_aug = add_interactions(X_te)
    m.fit(sc.fit_transform(X_tr_aug), y_bc)
    in_pred  = bc_to_log(m.predict(sc.transform(X_tr_aug)), lam)
    ext_pred = bc_to_log(m.predict(sc.transform(X_te_aug)), lam)
    return in_pred, ext_pred


def lin_ccc(y, yp):
    cov = np.cov(y, yp,ddof=0)[0, 1]
    return float((2*cov) / (y.var() + yp.var() + (y.mean()-yp.mean())**2))


def all_metrics(y, yp):
    r,_ = pearsonr(y, yp); rho,_ = spearmanr(y, yp)
    q2 = 1 - np.sum((y-yp)**2) / np.sum((y-y.mean())**2)
    rmse = float(np.sqrt(np.mean((y-yp)**2)))
    return {
        'Q2': round(q2, 4), 'CCC': round(lin_ccc(y, yp), 4),
        'r': round(r, 4), 'rho': round(rho, 4),
        'RMSE': round(rmse, 4), 'RMSE_real': round(np.exp(rmse), 3),
        'MAE': round(float(np.mean(np.abs(y-yp))), 4),
        'Bias': round(float(np.mean(yp-y)), 4),
    }


def gt_criteria(loo_q2, y_te, y_te_pred):
    r_te,_ = pearsonr(y_te, y_te_pred); r2_ext = r_te**2
    k  = float(np.sum(y_te_pred*y_te) / np.sum(y_te**2))
    kp = float(np.sum(y_te_pred*y_te) / np.sum(y_te_pred**2))
    ss_tot = np.sum((y_te - y_te.mean())**2)
    r0sq  = float(1 - np.sum((y_te - k*y_te_pred)**2)  / ss_tot)
    r0sqp = float(1 - np.sum((y_te - kp*y_te_pred)**2) / ss_tot)
    eps = 1e-9
    c1 = loo_q2 > 0.5
    c2 = r2_ext > 0.6
    c3 = (abs(r2_ext-r0sq)/(r2_ext+eps) < 0.1) or (abs(r2_ext-r0sqp)/(r2_ext+eps) < 0.1)
    c4 = (0.85 <= k <= 1.15) or (0.85 <= kp <= 1.15)
    c5 = abs(r0sq - r0sqp) < 0.3
    return {
        'r2_ext': round(r2_ext, 4), 'k': round(k, 4), 'kp': round(kp, 4),
        'R0sq': round(r0sq, 4), 'R0sqp': round(r0sqp, 4),
        'C1': c1, 'C2': c2, 'C3': c3, 'C4': c4, 'C5': c5,
        'PASS': c1 and c2 and c3 and c4 and c5,
    }


# ==============================================================================
#  INITIAL DATA PREPARATION (NOT LOOPED)
# ==============================================================================

# ══ STEP 1 — Standard cleaning ════════════════════════════════════════════════
banner("Step 1 — Standard data cleaning")
df = pd.read_excel(DATA_PATH)
TARGETS = [TARGET_FO, TARGET_HIX]
all_feats = [c for c in df.columns if c not in TARGETS]
df_clean = df.dropna(subset=all_feats).reset_index(drop=True)

df_model = df_clean.copy()
for f in LOG_TRANSFORM_FEATS:
    if f in df_model.columns and (df_model[f].values > 0).all():
        df_model[f] = np.log(df_model[f].values)
df_model.drop(columns=[c for c in COLLINEAR_DROP if c in df_model.columns], inplace=True)
feats_model = [f for f in all_feats if f in df_model.columns]

mask = np.ones(len(df_model), dtype=bool); mask[PRELIM_OUTLIER_ROWS] = False
df_model = df_model[mask].reset_index(drop=True)
print(f"  Loaded: {df.shape[0]} rows → cleaned: {len(df_model)} samples × {len(feats_model)} features")


# ══ STEP 2 — Kennard-Stone external test split ════════════════════════════════
banner("Step 2 — Kennard-Stone external test split")
X_all = df_model[feats_model].values
log_yfo_all = np.log(df_model[TARGET_FO].values)
log_yhix_all = np.log(df_model[TARGET_HIX].values)
raw_yfo_all = df_model[TARGET_FO].values
raw_yhix_all = df_model[TARGET_HIX].values

sc_ks = StandardScaler()
ks_idx = kennard_stone(sc_ks.fit_transform(X_all), n_select=9)
ks_mask = np.zeros(len(X_all), dtype=bool); ks_mask[ks_idx] = True

X_train_full, X_test = X_all[~ks_mask], X_all[ks_mask]
raw_yfo_tr_full = raw_yfo_all[~ks_mask]; raw_yhix_tr_full = raw_yhix_all[~ks_mask]
log_yfo_tr_full = log_yfo_all[~ks_mask]; log_yhix_tr_full = log_yhix_all[~ks_mask]
log_yfo_te = log_yfo_all[ks_mask]; log_yhix_te = log_yhix_all[ks_mask]

print(f"  Training pool: {X_train_full.shape[0]} samples")
print(f"  External test: {X_test.shape[0]} samples (Kennard-Stone — never touched)")


# ══ STEP 3 — Feature engineering ══════════════════════════════════════════════
banner("Step 3 — Feature engineering setup")
MATRIX_IDX = [feats_model.index(f) for f in MATRIX_FEATS]
FIBER_IDX  = [feats_model.index(f) for f in FIBER_FEATS]

n_inter = len(MATRIX_IDX) * len(FIBER_IDX)
print(f"  Interactions: {len(MATRIX_IDX)} matrix × {len(FIBER_IDX)} fiber = {n_inter}")
print(f"  Total features: {len(feats_model) + n_inter}")

# Define model kernels (constant across N_OUTLIERS loops)
kernel = (C(1.0, (1e-2, 1e2)) * RBF(1.0, (0.5, 20.0))
          + WhiteKernel(0.1, (1e-2, 1.0)))
gpr = GaussianProcessRegressor(
    kernel=kernel, alpha=0.1, n_restarts_optimizer=N_RESTARTS,
    normalize_y=True, random_state=RANDOM_SEED)
gpr_final = GaussianProcessRegressor(
    kernel=kernel, alpha=0.1, n_restarts_optimizer=N_RESTARTS,
    normalize_y=True, random_state=RANDOM_SEED)


def run_experiment(n_outliers_param):
    print(f"\n{'#'*70}")
    print(f"# Running experiment with N_OUTLIERS = {n_outliers_param}")
    print(f"{'#'*70}")

    # ══ STEP 4 — Williams plot outlier identification ═════════════════════════════
    banner("Step 4 — Williams plot outlier identification")
    bc_yfo_full, lam_fo_full = boxcox(raw_yfo_tr_full)

    # Initial LOO on training pool — first_order_rate
    preds_init = np.zeros(len(log_yfo_tr_full))
    for tr, te in LeaveOneOut().split(X_train_full):
        Xtr = add_interactions(X_train_full[tr])
        Xte = add_interactions(X_train_full[te])
        sc = StandardScaler(); m = copy.deepcopy(gpr)
        m.fit(sc.fit_transform(Xtr), bc_yfo_full[tr])
        preds_init[te] = m.predict(sc.transform(Xte)).ravel()[0]
    preds_init_log = bc_to_log(preds_init, lam_fo_full)
    residuals = log_yfo_tr_full - preds_init_log
    std_res = residuals / residuals.std()

    outlier_idx = np.argsort(np.abs(std_res))[-n_outliers_param:].tolist()
    outlier_idx_sorted = sorted(outlier_idx, key=lambda x: -abs(std_res[x]))

    print(f"  Removing top-{n_outliers_param} samples by |standardised LOO residual|:")
    for i in outlier_idx_sorted:
        print(f"    [{i}] log_rate={log_yfo_tr_full[i]:+7.2f}  pred={preds_init_log[i]:+7.2f}  "
              f"residual={residuals[i]:+6.2f}  std_res={std_res[i]:+6.2f}")


    # ══ STEP 5 — Final training set ═══════════════════════════════════════════════
    banner("Step 5 — Final training set")
    keep = np.ones(len(X_train_full), dtype=bool); keep[outlier_idx] = False
    X_train     = X_train_full[keep]
    raw_yfo_tr  = raw_yfo_tr_full[keep]
    raw_yhix_tr = raw_yhix_tr_full[keep]
    log_yfo_tr  = log_yfo_tr_full[keep]
    log_yhix_tr = log_yhix_tr_full[keep]
    n_train = X_train.shape[0]

    bc_yfo_tr,  lam_fo  = boxcox(raw_yfo_tr)
    bc_yhix_tr, lam_hix = boxcox(raw_yhix_tr)

    print(f"  Final training: {n_train} samples × {len(feats_model)} base features")
    print(f"  Box-Cox: λ_fo = {lam_fo:.4f}   λ_hix = {lam_hix:.4f}")


    # ══ STEP 6 — Final model evaluation ═══════════════════════════════════════════
    banner("Step 6 — Final model evaluation")

    RESULTS = {}
    for tname, bc_tr, log_tr, log_te, lam in [
        (TARGET_FO,  bc_yfo_tr,  log_yfo_tr,  log_yfo_te,  lam_fo),
        (TARGET_HIX, bc_yhix_tr, log_yhix_tr, log_yhix_te, lam_hix),
    ]:
        print(f"\n  ══ {tname} ══")
        preds_loo = loo_predictions(X_train, bc_tr, lam, gpr_final)
        m_loo = all_metrics(log_tr, preds_loo)

        in_pred, ext_pred = fit_and_predict(X_train, bc_tr, X_test, lam, gpr_final)
        r2_in = round(r2_score(log_tr, in_pred), 4)
        m_test = all_metrics(log_te, ext_pred)
        gt = gt_criteria(m_loo['Q2'], log_te, ext_pred)

        RESULTS[tname] = {
            'loo': m_loo, 'test': m_test, 'gt': gt, 'R2_in': r2_in,
            'preds_loo': preds_loo, 'preds_test': ext_pred, 'lam': lam,
        }

        print(f"    R² (in-sample)     = {r2_in}")
        print(f"    Q² (LOO)           = {m_loo['Q2']}")
        print(f"    CCC ★               = {m_loo['CCC']}  {'v ≥ 0.85' if m_loo['CCC']>=0.85 else 'BELOW 0.85'}")
        print(f"    Pearson r          = {m_loo['r']}")
        print(f"    Spearman ρ         = {m_loo['rho']}")
        print(f"    RMSE  (log / ×real)= {m_loo['RMSE']} / ×{m_loo['RMSE_real']}")
        print(f"    MAE                = {m_loo['MAE']}")
        print(f"    Bias               = {m_loo['Bias']}")
        print(f"\n    External test (n={X_test.shape[0]}):")
        print(f"      r²_ext = {gt['r2_ext']}    k = {gt['k']}    k' = {gt['kp']}")
        print(f"      R²₀    = {gt['R0sq']}    R²'₀ = {gt['R0sqp']}")
        print(f"\n    Golbraikh-Tropsha 2002:")
        for c in ['C1','C2','C3','C4','C5']:
            print(f"      {c}: {'v' if gt[c] else 'x'}")
        print(f"      OVERALL: {'PASS v' if gt['PASS'] else 'FAIL x'}")

    # ══ STEP 8 — Save outputs ═════════════════════════════════════════════════
    banner("Step 8 — Save outputs")
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    pred_df = pd.DataFrame({
        'log_actual_fo':    log_yfo_tr,
        'log_pred_fo_loo':  RESULTS[TARGET_FO]['preds_loo'],
        'residual_fo':      log_yfo_tr - RESULTS[TARGET_FO]['preds_loo'],
        'log_actual_hix':   log_yhix_tr,
        'log_pred_hix_loo': RESULTS[TARGET_HIX]['preds_loo'],
        'residual_hix':     log_yhix_tr - RESULTS[TARGET_HIX]['preds_loo'],
    })
    p1 = os.path.join(OUTPUT_DIR, f'final_robust_predictions_N{n_outliers_param}.csv')
    pred_df.to_csv(p1, index=True); print(f"  {p1}")

    ext_df = pd.DataFrame({
        'log_actual_fo_test':  log_yfo_te,
        'log_pred_fo_test':    RESULTS[TARGET_FO]['preds_test'],
        'log_actual_hix_test': log_yhix_te,
        'log_pred_hix_test':   RESULTS[TARGET_HIX]['preds_test'],
    })
    p2 = os.path.join(OUTPUT_DIR, f'final_robust_external_test_N{n_outliers_param}.csv')
    ext_df.to_csv(p2, index=True); print(f"  {p2}")

    metrics_rows = []
    for tname in [TARGET_FO, TARGET_HIX]:
        r = RESULTS[tname]
        metrics_rows.append({
            'target': tname, 'n_train': n_train, 'n_test': X_test.shape[0],
            'box_cox_lambda':   round(r['lam'], 4),
            'R2_in_sample':     r['R2_in'],
            'Q2_LOO':           r['loo']['Q2'],
            'CCC_LOO':          r['loo']['CCC'],
            'Pearson_r_LOO':    r['loo']['r'],
            'Spearman_rho_LOO': r['loo']['rho'],
            'RMSE_log':         r['loo']['RMSE'],
            'RMSE_real_x':      r['loo']['RMSE_real'],
            'MAE_log':          r['loo']['MAE'],
            'Bias_LOO':         r['loo']['Bias'],
            'r2_ext':           r['gt']['r2_ext'],
            'GT_k':             r['gt']['k'],
            'GT_k_prime':       r['gt']['kp'],
            'GT_R0sq':          r['gt']['R0sq'],
            'GT_R0sq_prime':    r['gt']['R0sqp'],
            'GT_C1':            r['gt']['C1'],
            'GT_C2':            r['gt']['C2'],
            'GT_C3':            r['gt']['C3'],
            'GT_C4':            r['gt']['C4'],
            'GT_C5':            r['gt']['C5'],
            'GT_OVERALL_PASS':  r['gt']['PASS'],
            'permutation_p':    0.0 # Permutation test skipped
        })
    met_df = pd.DataFrame(metrics_rows)
    p3 = os.path.join(OUTPUT_DIR, f'final_robust_metrics_N{n_outliers_param}.csv')
    met_df.to_csv(p3, index=False); print(f"  {p3}")

    p4 = os.path.join(OUTPUT_DIR, f'final_robust_metrics_N{n_outliers_param}.xlsx')
    try:
        with pd.ExcelWriter(p4, engine='openpyxl') as writer:
            met_df.to_excel(writer, sheet_name='Metrics', index=False)
            pred_df.to_excel(writer, sheet_name='LOO Predictions')
            ext_df.to_excel(writer, sheet_name='External Test')
        print(f"  {p4}")
    except Exception as e:
        print(f"  Excel skipped for N_OUTLIERS={n_outliers_param}: {e}")


# ==============================================================================
#  LOOP THROUGH N_OUTLIERS VALUES
# ==============================================================================
for n_outliers_val in range(5, 16): # Loop from 5 to 15 inclusive
    run_experiment(n_outliers_val)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

# --- 1. Aggregate Data from your N_OUTLIERS loop ---
n_range = range(5, 16)
plot_data = []

for n in n_range:
    file_path = f'final_robust_metrics_N{n}.csv'
    if os.path.exists(file_path):
        df_met = pd.read_csv(file_path)
        for _, row in df_met.iterrows():
            plot_data.append({
                'N': n,
                'Target': row['target'],
                'Gap': row['R2_in_sample'] - row['Q2_LOO'],
                'CCC': row['CCC_LOO'],
                'r2_ext': row['r2_ext']
            })

df_plot = pd.DataFrame(plot_data)
# --- 2. Create the Visual Proof ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
targets = df_plot['Target'].unique()
colors = {'first_order_rate': '#1f77b4', 'hixson_rate': '#ff7f0e'}
for target in targets:
    subset = df_plot[df_plot['Target'] == target]

    # Subplot 1: Stability Gap (Lower is better)
    axes[0].plot(subset['N'], subset['Gap'], marker='o', label=target, color=colors[target])
    axes[0].set_title('N vs Stability Gap ($R^2 - Q^2$)')
    axes[0].set_ylabel('Gap Magnitude')
    axes[0].axvline(9, color='red', linestyle='--', alpha=0.5)

    # Subplot 2: CCC (Higher is better)
    axes[1].plot(subset['N'], subset['CCC'], marker='s', label=target, color=colors[target])
    axes[1].set_title('N vs Concordance (CCC)')
    axes[1].set_ylabel('CCC Value')
    axes[1].axhline(0.85, color='green', linestyle=':', label='Threshold (0.85)')
    axes[1].axvline(9, color='red', linestyle='--', alpha=0.5)

    # Subplot 3: External Predictivity (Higher is better)
    axes[2].plot(subset['N'], subset['r2_ext'], marker='^', label=target, color=colors[target])
    axes[2].set_title('N vs External $r^2$')
    axes[2].set_ylabel('External $r^2$')
    axes[2].axhline(0.60, color='green', linestyle=':', label='C2 Threshold (0.60)')
    axes[2].axvline(9, color='red', linestyle='--', alpha=0.5)

for ax in axes:
    ax.set_xlabel('Number of Outliers Removed (N)')
    ax.legend()
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.suptitle('Proof of Optimal N Selection (N=9)', y=1.05, fontsize=14, fontweight='bold')
plt.savefig("outlier.svg")
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

def calculate_rmsep(actual, predicted):
    return np.sqrt(np.mean((actual - predicted)**2))

n_range = range(5, 16)
targets = ['first_order_rate', 'hixson_rate']
plot_data = []

for n in n_range:
    # 1. Extract LOO Metrics from metrics files
    metrics_file = f'final_robust_metrics_N{n}.csv'
    if os.path.exists(metrics_file):
        df_met = pd.read_csv(metrics_file)
        # 2. Extract Predictions from external test files to calculate RMSEP
        ext_file = f'final_robust_external_test_N{n}.csv'
        if os.path.exists(ext_file):
            df_ext = pd.read_csv(ext_file)

            for target in targets:
                # Get metrics from LOO
                row = df_met[df_met['target'] == target].iloc[0]
                rmse_loo = row['RMSE_log']
                mae_loo = row['MAE_log']

                # Calculate RMSEP (External RMSE)
                if target == 'first_order_rate':
                    rmsep = calculate_rmsep(df_ext['log_actual_fo_test'], df_ext['log_pred_fo_test'])
                else:
                    rmsep = calculate_rmsep(df_ext['log_actual_hix_test'], df_ext['log_pred_hix_test'])

                plot_data.append({
                    'N': n,
                    'Target': target,
                    'RMSE_LOO': rmse_loo,
                    'MAE_LOO': mae_loo,
                    'RMSEP': rmsep
                })

df_res = pd.DataFrame(plot_data)

# --- Visualization ---
fig, axes = plt.subplots(1, 2, figsize=(18, 6), sharey=True)

for i, target in enumerate(targets):
    subset = df_res[df_res['Target'] == target]
    axes[i].plot(subset['N'], subset['RMSE_LOO'], marker='o', label='RMSE (LOO)', linewidth=2)
    axes[i].plot(subset['N'], subset['MAE_LOO'], marker='s', label='MAE (LOO)', linewidth=2)
    axes[i].plot(subset['N'], subset['RMSEP'], marker='^', label='RMSEP (External)', linestyle='--', linewidth=2)

    axes[i].set_title(f'Error Metrics vs N_OUTLIERS\n({target.replace("_", " ").title()})', fontsize=14, fontweight='bold')
    axes[i].set_xlabel('Number of Outliers Removed (N)', fontsize=12)
    axes[i].set_ylabel('Error Value (Log Scale)', fontsize=12)
    axes[i].grid(True, alpha=0.3)
    axes[i].legend()

    # Mark the sweet spot at N=9
    axes[i].axvline(9, color='red', linestyle=':', alpha=0.5)
    axes[i].annotate('N=9', xy=(9, axes[i].get_ylim()[0]), color='red', fontweight='bold')

plt.tight_layout()
plt.savefig('error_metrics_plot.png')
plt.show()

# Print summary table
print(df_res.sort_values(['Target', 'N']).to_string(index=False))